# Winword DOCX file assembly lineage

## PowerShell
Get-ChildItem -Path "Z:\TB" -Filter *.docx -Recurse -ErrorAction SilentlyContinue | Where-Object { $_.LastWriteTime -ge (Get-Date).AddDays(-60) } | Select-Object LastWriteTime, Name, Length, FullName | Export-Csv -Path ".\Recently_Modified_Files2.csv" -NoTypeInformation

# Third version: First, it scans the folder and its subfolders for .docx files, collecting metadata and creating a text block matrix in the process.

The collection also writes `<RUN_NAME>_text_cache.json.gz` for future target-passage searches. It stores current body/table/text-box text, normalized words, paragraph locations and file fingerprints. Deleted/moved-from text is excluded; inserted/moved-to text is included. XML paragraph ordinals are not Word page numbers. The cache covers only files included by the existing scan filters. Set `WRITE_TEXT_CACHE = False` to skip writing it; any older cache then remains an older snapshot.


In [ ]:
# 3rd version: This:
import os
import re
import csv
import json
import gzip
import tempfile
import hashlib
import zipfile
import difflib
from collections import OrderedDict
from functools import lru_cache
from bisect import bisect_left
from time import perf_counter
import xml.etree.ElementTree as ET

from datetime import datetime, timedelta, timezone


# ============================================================
# INPUT / OUTPUT
# ============================================================

#ROOT_DIR = r"Z:\TB\TEXT-FIGURE formulation Preparation-202604-\Working_File"
ROOT_DIR = r"Z:\TB"

OUTPUT_DIR = ROOT_DIR

RUN_NAME = "Working_File2nd"

NODES_FILE_NAME = f"{RUN_NAME}_nodes.csv"
EDGES_FILE_NAME = f"{RUN_NAME}_edges.csv"
DOT_FILE_NAME   = f"{RUN_NAME}_history.dot"

NODES_FILE = os.path.join(OUTPUT_DIR, NODES_FILE_NAME)
EDGES_FILE = os.path.join(OUTPUT_DIR, EDGES_FILE_NAME)
DOT_FILE   = os.path.join(OUTPUT_DIR, DOT_FILE_NAME)

# Companion snapshot for future on-demand target passage searches.
# No extra all-pairs analysis; the normal CSV/DOT schemas are unchanged.
WRITE_TEXT_CACHE = True
TEXT_CACHE_FILE = os.path.join(OUTPUT_DIR, f"{RUN_NAME}_text_cache.json.gz")
TEXT_CACHE_SCHEMA_VERSION = 1
TEXT_EXTRACTION_VERSION = "body-visible-v1"
TEXT_NORMALIZATION_VERSION = "lowercase-unicode-words-v1"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# ANALYSIS CONFIG
# ============================================================

DAYS_AGO = 120
MAX_PARENT_DAYS = 90
MIN_PARAGRAPH_CHARS = 35
SHINGLE_SIZE = 7
MIN_WORDS_FOR_SHINGLES = 40

MIN_EDGE_SCORE = 0.42
MAX_PARENTS = 3
PARENT_SCORE_MARGIN = 0.10

# Enhanced mode adds explicitly uncertain candidates; Score remains unchanged.
# False restores legacy candidate selection (empty-text matches are still guarded).
ENHANCED_LINEAGE = True
MAX_FALLBACK_PARENTS = 2
MAX_FALLBACK_CHECKS_PER_DOCUMENT = 6
MAX_ENHANCED_DIFF_TOKENS = 12000
MIN_SHARED_SHINGLES = 20
MIN_SHINGLE_CONTAINMENT = 0.70
MIN_SHINGLE_TOKEN_SIMILARITY = 0.35
MIN_FILENAME_TOKEN_SIMILARITY = 0.30
ENHANCED_DETAIL_THRESHOLD = 0.65


# ============================================================
# NEW: DETAILED DIFF FOR VERY SIMILAR PAIRS
# ============================================================

DETAIL_DIFF_THRESHOLD = 0.95
DETAIL_DIFF_SAME_WORDCOUNT_MIN_SCORE = 0.90
DIFF_SAMPLE_CHARS = 180


# ============================================================
# WORD XML CONSTANTS
# ============================================================

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
W_P = f"{{{W_NS}}}p"
W_T = f"{{{W_NS}}}t"


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(text):
    """
    Broad lineage normalization:
    lowercase, punctuation-insensitive, whitespace-normalized.
    """
    text = text.lower()
    text = text.replace("\xa0", " ")
    text = re.sub(r"[^\w]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_for_diff(text):
    """
    Lighter normalization for detailed diff:
    preserve case and punctuation; normalize whitespace only.
    """
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


@lru_cache(maxsize=256)
def tokenize_for_diff(text):
    """
    Tokenize words and punctuation separately.
    """
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def short_hash(text):
    return hashlib.blake2b(
        text.encode("utf-8", errors="ignore"),
        digest_size=8
    ).hexdigest()


def full_file_hash(filepath):
    h = hashlib.sha256()

    try:
        with open(filepath, "rb") as f:
            while True:
                chunk = f.read(1024 * 1024)
                if not chunk:
                    break
                h.update(chunk)

        return h.hexdigest()

    except OSError:
        return "ERROR"


# ============================================================
# DOCX EXTRACTION
# ============================================================

def extract_body_paragraph_records(xml_bytes):
    """Current body/table/text-box text; locations are 1-based XML paragraph ordinals.

    Includes insertions and moved-to text; excludes deletions and moved-from text.
    Empty/deleted paragraphs still occupy XML ordinals. These are not page numbers.
    Nested text-box paragraphs are emitted once, separately from their container.
    """
    w = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'
    root = ET.fromstring(xml_bytes)
    ordinals = {id(p): i for i, p in enumerate(root.iter(w+'p'), 1)}
    records = []

    def pieces(element):
        if element.tag in {w+'del', w+'moveFrom'}:
            return []
        if element.tag == w+'t':
            return [element.text or '']
        if element.tag in {w+'tab', w+'br', w+'cr'}:
            return [' ']
        result = []
        for child in element:
            if child.tag != w+'p':
                result.extend(pieces(child))
        return result

    def walk(element):
        if element.tag in {w+'del', w+'moveFrom'}:
            return
        if element.tag == w+'p':
            text = re.sub(r'\s+', ' ', ''.join(pieces(element)).replace('\xa0', ' ')).strip()
            if text:
                records.append({'xml_paragraph_index': ordinals[id(element)], 'text': text})
        for child in element:
            walk(child)
    walk(root)
    return records


def extract_paragraphs_from_xml(xml_bytes):
    return [record['text'] for record in extract_body_paragraph_records(xml_bytes)]


def extract_core_metadata(docx):
    metadata = {
        "created": "",
        "modified": "",
        "revision": "",
        "last_modified_by": "",
        "creator": "",
    }

    if "docProps/core.xml" not in docx.namelist():
        return metadata

    try:
        root = ET.fromstring(docx.read("docProps/core.xml"))

        for elem in root.iter():
            tag = elem.tag.split("}")[-1]

            if tag == "created":
                metadata["created"] = elem.text or ""

            elif tag == "modified":
                metadata["modified"] = elem.text or ""

            elif tag == "revision":
                metadata["revision"] = elem.text or ""

            elif tag == "lastModifiedBy":
                metadata["last_modified_by"] = elem.text or ""

            elif tag == "creator":
                metadata["creator"] = elem.text or ""

    except Exception:
        pass

    return metadata


def inspect_docx(filepath):
    result = {
        "paragraphs": [],
        "paragraph_records": [],
        "extraction_status": "error",
        "extraction_error": "",
        "normalized_paragraphs": [],
        "diff_paragraphs": [],
        "paragraph_hashes": set(),
        "shingles": set(),
        "normalized_text": "",
        "diff_text": "",
        "content_hash": "",
        "word_count": 0,
        "tracked_changes": False,
        "comments": False,
        "metadata": {},
    }

    try:
        with zipfile.ZipFile(filepath, "r") as docx:

            names = set(docx.namelist())

            result["metadata"] = extract_core_metadata(docx)
            result["comments"] = "word/comments.xml" in names

            if "word/document.xml" not in names:
                result["extraction_error"] = "Missing word/document.xml"
                return result

            doc_xml = docx.read("word/document.xml")

            result["tracked_changes"] = (
                b"<w:ins" in doc_xml
                or b"<w:del" in doc_xml
                or b"<w:moveFrom" in doc_xml
                or b"<w:moveTo" in doc_xml
            )

            records = extract_body_paragraph_records(doc_xml)
            result["paragraph_records"] = records
            paragraphs = [record['text'] for record in records]
            result["extraction_status"] = "ok" if paragraphs else "empty"
            result["paragraphs"] = paragraphs

            normalized_paragraphs = []
            diff_paragraphs = []

            for para in paragraphs:
                normalized = normalize_text(para)
                diff_para = normalize_for_diff(para)

                if normalized:
                    normalized_paragraphs.append(normalized)

                if diff_para:
                    diff_paragraphs.append(diff_para)

            result["normalized_paragraphs"] = normalized_paragraphs
            result["diff_paragraphs"] = diff_paragraphs

            normalized_text = "\n".join(normalized_paragraphs)
            diff_text = "\n".join(diff_paragraphs)

            result["normalized_text"] = normalized_text
            result["diff_text"] = diff_text

            words = normalize_text(normalized_text).split()
            result["word_count"] = len(words)

            # Empty extraction is not evidence of identical document text.
            result["content_hash"] = (hashlib.sha256(
                normalized_text.encode("utf-8", errors="ignore")
            ).hexdigest() if normalized_text else "")

            paragraph_hashes = set()

            for para in normalized_paragraphs:
                if len(para) >= MIN_PARAGRAPH_CHARS:
                    paragraph_hashes.add(short_hash(para))

            result["paragraph_hashes"] = paragraph_hashes

            shingles = set()

            if len(words) >= MIN_WORDS_FOR_SHINGLES:
                for i in range(len(words) - SHINGLE_SIZE + 1):
                    shingle = " ".join(
                        words[i:i + SHINGLE_SIZE]
                    )
                    shingles.add(short_hash(shingle))

            result["shingles"] = shingles

    except Exception as exc:
        result["extraction_status"] = "error"
        result["extraction_error"] = str(exc)
        print("ERROR:", filepath, exc)

    return result


# ============================================================
# FAST SIMILARITY
# ============================================================

def safe_ratio(a, b):
    if not b:
        return 0.0
    return a / b


def intersection_size(a, b):
    if len(a) < len(b):
        return sum(1 for x in a if x in b)

    return sum(1 for x in b if x in a)


def jaccard(a, b):
    if not a or not b:
        return 0.0

    intersection = intersection_size(a, b)
    union = len(a) + len(b) - intersection

    if not union:
        return 0.0

    return intersection / union


# Per-run, bounded caches. Keys retain exact extracted text (including case and
# punctuation for detailed diffs); filenames, folders and chronology are never cached.
BUILDER_PAIR_CACHE_SIZE = 2048
_similarity_cache = OrderedDict()
_detail_cache = OrderedDict()
_shingle_cache = OrderedDict()
_builder_cache_hits = {'similarity': 0, 'detail': 0, 'shingles': 0}


def cached_builder_pair(cache, key, calculate, kind):
    if key in cache:
        cache.move_to_end(key)
        _builder_cache_hits[kind] += 1
        result = cache[key]
    else:
        result = calculate()
        cache[key] = result
        if len(cache) > BUILDER_PAIR_CACHE_SIZE:
            cache.popitem(last=False)
    return result.copy() if isinstance(result, dict) else result


def shared_shingle_count(old, new):
    if 'normalized_text' not in old or 'normalized_text' not in new:
        return intersection_size(old['shingles'], new['shingles'])
    key = (old['normalized_text'], new['normalized_text'], SHINGLE_SIZE, MIN_WORDS_FOR_SHINGLES)
    return cached_builder_pair(_shingle_cache, key,
        lambda: intersection_size(old['shingles'], new['shingles']), 'shingles')


def compare_documents(old, new):
    if 'normalized_text' not in old or 'normalized_text' not in new:
        return compare_documents_uncached(old, new)
    key = (old['normalized_text'], new['normalized_text'], MIN_PARAGRAPH_CHARS,
           SHINGLE_SIZE, MIN_WORDS_FOR_SHINGLES)
    return cached_builder_pair(_similarity_cache, key,
        lambda: compare_documents_uncached(old, new), 'similarity')


def compare_documents_uncached(old, new):
    old_p = old["paragraph_hashes"]
    new_p = new["paragraph_hashes"]

    common_paragraphs = intersection_size(old_p, new_p)

    old_in_new = safe_ratio(
        common_paragraphs,
        len(old_p)
    )

    new_in_old = safe_ratio(
        common_paragraphs,
        len(new_p)
    )

    para_union = len(old_p) + len(new_p) - common_paragraphs
    para_jaccard = common_paragraphs / para_union if para_union else 0.0
    common_shingles = shared_shingle_count(old, new)
    shingle_union = len(old['shingles']) + len(new['shingles']) - common_shingles
    shingle_jaccard = common_shingles / shingle_union if shingle_union else 0.0

    old_words = old["word_count"]
    new_words = new["word_count"]

    if max(old_words, new_words):
        size_similarity = (
            min(old_words, new_words)
            / max(old_words, new_words)
        )
    else:
        size_similarity = 0

    relationship = "weak similarity"

    if (
        old_words > 0 and new_words > 0
        and old["content_hash"]
        and old["content_hash"] == new["content_hash"]
    ):
        relationship = "same normalized text / renamed copy"
        score = 1.0

    else:

        if (
            new_in_old >= 0.80
            and new_words < old_words * 0.75
        ):
            relationship = "excerpt / extracted from"

        elif (
            old_in_new >= 0.80
            and new_words > old_words * 1.20
        ):
            relationship = "expanded / copied into"

        elif (
            old_in_new >= 0.65
            and new_in_old >= 0.65
        ):
            relationship = "revision"

        elif max(old_in_new, new_in_old) >= 0.60:
            relationship = "partial reuse"

        score = (
            0.50 * max(old_in_new, new_in_old)
            + 0.20 * para_jaccard
            + 0.20 * shingle_jaccard
            + 0.10 * size_similarity
        )

    return {
        "score": score,
        "relationship": relationship,
        "old_in_new": old_in_new,
        "new_in_old": new_in_old,
        "paragraph_jaccard": para_jaccard,
        "shingle_jaccard": shingle_jaccard,
        "common_paragraphs": common_paragraphs,
    }


# ============================================================
# NEW: DETAILED PAIRWISE DIFF
# ============================================================

def truncate_sample(text, limit=DIFF_SAMPLE_CHARS):
    text = normalize_for_diff(text)

    if len(text) <= limit:
        return text

    return text[:limit - 3] + "..."


def detailed_text_diff(old, new):
    key = (old['diff_text'], new['diff_text'], tuple(old['diff_paragraphs']),
           tuple(new['diff_paragraphs']), DIFF_SAMPLE_CHARS)
    return cached_builder_pair(_detail_cache, key,
        lambda: detailed_text_diff_uncached(old, new), 'detail')


def detailed_text_diff_uncached(old, new):
    """
    Detailed diff for near-identical pairs.

    Diff_Edit_Blocks is NOT Microsoft's Word Compare revision count.
    It is a SequenceMatcher edit-block count and can differ from Word.
    """

    old_paras = old["diff_paragraphs"]
    new_paras = new["diff_paragraphs"]

    old_text = old["diff_text"]
    new_text = new["diff_text"]

    exact_visible_equal = (old_text == new_text)
    if exact_visible_equal and old_paras == new_paras:
        # Identical text needs no SequenceMatcher pass; retain all existing fields.
        count = len(tokenize_for_diff(old_text))
        result = {key: 0 for key in blank_detailed_diff()}
        result.update(detail_diff_run=True, exact_visible_text_equal=True,
                      diff_class='exact visible text', token_similarity=1.0,
                      text_delta_pct=0.0, old_token_count=count, new_token_count=count,
                      changed_old_sample='', changed_new_sample='')
        return result

    # Paragraph-level comparison
    para_matcher = difflib.SequenceMatcher(
        None,
        old_paras,
        new_paras,
        autojunk=False
    )

    para_inserted = 0
    para_deleted = 0
    para_replaced_old = 0
    para_replaced_new = 0
    paragraph_edit_blocks = 0

    changed_old_samples = []
    changed_new_samples = []

    for tag, i1, i2, j1, j2 in para_matcher.get_opcodes():

        if tag == "equal":
            continue

        paragraph_edit_blocks += 1

        if tag == "insert":
            para_inserted += (j2 - j1)

        elif tag == "delete":
            para_deleted += (i2 - i1)

        elif tag == "replace":
            para_replaced_old += (i2 - i1)
            para_replaced_new += (j2 - j1)

        if len(changed_old_samples) < 3 and i2 > i1:
            changed_old_samples.append(
                truncate_sample(
                    " ".join(old_paras[i1:i2])
                )
            )

        if len(changed_new_samples) < 3 and j2 > j1:
            changed_new_samples.append(
                truncate_sample(
                    " ".join(new_paras[j1:j2])
                )
            )

    changed_paragraphs_est = (
        para_inserted
        + para_deleted
        + max(
            para_replaced_old,
            para_replaced_new
        )
    )

    # Token-level comparison
    old_tokens = tokenize_for_diff(old_text)
    new_tokens = tokenize_for_diff(new_text)

    token_matcher = difflib.SequenceMatcher(
        None,
        old_tokens,
        new_tokens,
        autojunk=False
    )

    inserted_tokens = 0
    deleted_tokens = 0
    replaced_old_tokens = 0
    replaced_new_tokens = 0
    diff_edit_blocks = 0

    for tag, i1, i2, j1, j2 in token_matcher.get_opcodes():

        if tag == "equal":
            continue

        diff_edit_blocks += 1

        old_len = i2 - i1
        new_len = j2 - j1

        if tag == "insert":
            inserted_tokens += new_len

        elif tag == "delete":
            deleted_tokens += old_len

        elif tag == "replace":
            replaced_old_tokens += old_len
            replaced_new_tokens += new_len

    changed_tokens_est = (
        inserted_tokens
        + deleted_tokens
        + max(
            replaced_old_tokens,
            replaced_new_tokens
        )
    )

    token_denominator = max(
        len(old_tokens),
        len(new_tokens),
        1
    )

    text_delta_pct = (
        changed_tokens_est
        / token_denominator
        * 100.0
    )

    token_similarity = token_matcher.ratio()

    if exact_visible_equal:
        diff_class = "exact visible text"
    elif text_delta_pct <= 0.10:
        diff_class = "tiny edit"
    elif text_delta_pct <= 1.00:
        diff_class = "minor revision"
    elif text_delta_pct <= 5.00:
        diff_class = "moderate revision"
    else:
        diff_class = "substantial revision"

    return {
        "detail_diff_run": True,
        "exact_visible_text_equal": exact_visible_equal,
        "diff_class": diff_class,

        "paragraph_edit_blocks": paragraph_edit_blocks,
        "changed_paragraphs_est": changed_paragraphs_est,
        "paragraphs_inserted": para_inserted,
        "paragraphs_deleted": para_deleted,
        "paragraphs_replaced_old": para_replaced_old,
        "paragraphs_replaced_new": para_replaced_new,

        "diff_edit_blocks": diff_edit_blocks,
        "inserted_tokens": inserted_tokens,
        "deleted_tokens": deleted_tokens,
        "replaced_old_tokens": replaced_old_tokens,
        "replaced_new_tokens": replaced_new_tokens,
        "changed_tokens_est": changed_tokens_est,

        "token_similarity": token_similarity,
        "text_delta_pct": text_delta_pct,

        "old_token_count": len(old_tokens),
        "new_token_count": len(new_tokens),

        "changed_old_sample": " || ".join(
            changed_old_samples
        ),
        "changed_new_sample": " || ".join(
            changed_new_samples
        ),
    }


def blank_detailed_diff():
    return {
        "detail_diff_run": False,
        "exact_visible_text_equal": "",
        "diff_class": "",
        "paragraph_edit_blocks": "",
        "changed_paragraphs_est": "",
        "paragraphs_inserted": "",
        "paragraphs_deleted": "",
        "paragraphs_replaced_old": "",
        "paragraphs_replaced_new": "",
        "diff_edit_blocks": "",
        "inserted_tokens": "",
        "deleted_tokens": "",
        "replaced_old_tokens": "",
        "replaced_new_tokens": "",
        "changed_tokens_est": "",
        "token_similarity": "",
        "text_delta_pct": "",
        "old_token_count": "",
        "new_token_count": "",
        "changed_old_sample": "",
        "changed_new_sample": "",
    }


def should_run_detailed_diff(edge):
    score = edge["score"]

    if score >= DETAIL_DIFF_THRESHOLD:
        return True

    parent_words = edge["parent"]["word_count"]
    child_words = edge["child"]["word_count"]

    if (
        parent_words == child_words
        and score >= DETAIL_DIFF_SAME_WORDCOUNT_MIN_SCORE
    ):
        return True

    return False


def filename_evidence(filename):
    """Keep distinguishing words (e.g. CL/EL); remove only valid dates and tail versions."""
    stem = os.path.splitext(filename)[0].casefold()
    version_match = re.search(r'(?:[_ .-]+v\d+)+$', stem)
    versions = tuple(int(x) for x in re.findall(r'v(\d+)', version_match.group(0))) if version_match else ()
    if version_match:
        stem = stem[:version_match.start()]
    dates = []

    def remove_date(match):
        for fmt in ('%Y%m%d', '%d%m%Y'):
            try:
                value = datetime.strptime(match.group(), fmt)
                if 1990 <= value.year <= 2100:
                    dates.append(value)
                    return ' '
            except ValueError:
                pass
        return match.group()

    stem = re.sub(r'(?<!\d)\d{8}(?!\d)', remove_date, stem)
    family = ' '.join(re.findall(r'[^\W_]+', stem, flags=re.UNICODE))
    # Generic names such as "chapter1" alone are not enough for a fallback.
    if len(family) < 12 or len(family.split()) < 3:
        family = ''
    return {'family': family, 'versions': versions, 'date': max(dates) if dates else None}


def document_state_key(document):
    digest = document.get('binary_hash', '')
    return ('binary', digest) if digest and digest != 'ERROR' else ('file', document['id'])


def enhanced_pair_evidence(old, new):
    a = old.get('filename_evidence') or filename_evidence(old['filename'])
    b = new.get('filename_evidence') or filename_evidence(new['filename'])
    same_family = bool(a['family'] and a['family'] == b['family'])
    same_folder = os.path.normcase(old['folder']) == os.path.normcase(new['folder'])
    version = 'unknown'
    if a['versions'] and b['versions']:
        version = 'forward' if b['versions'] > a['versions'] else 'backward' if b['versions'] < a['versions'] else 'same'
    date_order = 'unknown'
    if a['date'] and b['date']:
        date_order = 'forward' if b['date'] > a['date'] else 'backward' if b['date'] < a['date'] else 'same'
    gap = (new['mtime'] - old['mtime']).total_seconds()/3600
    chronology = 'conflict' if gap < 0 or (same_family and (version == 'backward' or date_order == 'backward')) else 'equal timestamps' if gap == 0 else 'forward'
    common = shared_shingle_count(old, new)
    old_containment = safe_ratio(common, len(old['shingles']))
    new_containment = safe_ratio(common, len(new['shingles']))
    name_candidate = (same_family and chronology != 'conflict'
                      and (gap > 0 or version == 'forward' or date_order == 'forward')
                      and (same_folder or version == 'forward' or date_order == 'forward'))
    shingle_candidate = (common >= MIN_SHARED_SHINGLES and max(old_containment, new_containment) >= MIN_SHINGLE_CONTAINMENT)
    # Metadata priority is only used to bound which expensive comparisons run.
    priority = (0.40 * same_family + 0.20 * (version == 'forward') + 0.10 * same_folder
                + 0.10 * (date_order == 'forward') + 0.20 / (1 + max(gap, 0)/24))
    return dict(filename_family_match=same_family, version_progression=version,
                filename_date_order=date_order, same_folder=same_folder,
                chronology_evidence=chronology, old_shingle_containment=old_containment,
                new_shingle_containment=new_containment, common_shingles=common,
                name_candidate=name_candidate, shingle_candidate=shingle_candidate,
                metadata_priority=priority)


def bounded_detail(edge):
    old, new = edge['parent'], edge['child']
    count = max(len(tokenize_for_diff(old['diff_text'])), len(tokenize_for_diff(new['diff_text'])))
    if count > MAX_ENHANCED_DIFF_TOKENS:
        return None
    return detailed_text_diff(old, new)


def parent_text_state_key(document):
    # Keep case and punctuation edits distinct. Broad normalized hashes would
    # merge these revisions; a binary hash would count metadata-only saves twice.
    text = document.get('diff_text', '')
    return ('visible_text', text) if text else document_state_key(document)


def select_enhanced_parents(new, candidates, fallback):
    """Retain distinct edited parent texts plus one same-extracted-text copy edge.

    Score remains the original content score. Candidate_Score is a separate
    heuristic for optional inferred relationships, not a probability.
    """
    new_key = parent_text_state_key(new)
    duplicate_edges = [e for e in candidates if parent_text_state_key(e['parent']) == new_key]
    content = [e for e in candidates if parent_text_state_key(e['parent']) != new_key]

    def unique_states(items, sort_key):
        result = []
        seen = set()
        for edge in sorted(items, key=sort_key, reverse=True):
            key = parent_text_state_key(edge['parent'])
            if key not in seen:
                seen.add(key)
                result.append(edge)
        return result

    # Apply the score margin only among edited texts: a 1.0 copy must not
    # exclude a genuine 0.85 predecessor or use one of its parent slots.
    content = unique_states(content, lambda e: (e['score'], e['same_folder'], e['parent']['mtime'], e['parent']['id']))
    if content:
        content = [e for e in content if e['score'] >= content[0]['score'] - PARENT_SCORE_MARGIN][:MAX_PARENTS]
    for edge in content + duplicate_edges:
        edge.update(evidence_type='content', candidate_score='', candidate_reason='Original content similarity threshold met.', uncertain=False)

    # Verify only a bounded number of distinct plausible fallback parents.
    fallback = unique_states(fallback, lambda e: (e['metadata_priority'] + max(e['old_shingle_containment'], e['new_shingle_containment']), e['parent']['mtime'], e['parent']['id']))
    verified = []
    for edge in fallback[:MAX_FALLBACK_CHECKS_PER_DOCUMENT]:
        detail = bounded_detail(edge)
        if detail is None:
            continue  # Unverified name-only matches remain available as filename hints in Dash.
        token_similarity = detail['token_similarity']
        if edge['shingle_candidate'] and token_similarity >= MIN_SHINGLE_TOKEN_SIMILARITY:
            kind = 'shingle_candidate'
            reason = 'Strong directional shingle reuse, verified by token comparison; exact paragraphs may have changed.'
            confidence = .65 * max(edge['old_shingle_containment'], edge['new_shingle_containment']) + .35 * token_similarity
        elif edge['name_candidate'] and token_similarity >= MIN_FILENAME_TOKEN_SIMILARITY:
            kind = 'filename_candidate'
            reason = ('Matching filename family; version=' + edge['version_progression']
                      + '; filename date=' + edge['filename_date_order']
                      + '; chronology=' + edge['chronology_evidence']
                      + '; ' + ('same folder' if edge['same_folder'] else 'different folders')
                      + '. Token comparison supports possible rewriting; exact parentage is uncertain.')
            confidence = .65 * edge['metadata_priority'] + .35 * token_similarity
        else:
            continue
        edge.update(detail)
        edge.update(evidence_type=kind, candidate_score=round(confidence, 6), candidate_reason=reason,
                    uncertain=True, relationship='possible rewritten version' if kind == 'filename_candidate' else 'possible edited / reused content')
        verified.append(edge)
    verified.sort(key=lambda e: (e['candidate_score'], e['parent']['mtime'], e['parent']['id']), reverse=True)
    # Additional evidence has its own small allowance so it cannot displace strong content parents.
    selected = content + verified[:MAX_FALLBACK_PARENTS]
    if duplicate_edges:
        selected.append(max(duplicate_edges, key=lambda e: (e['same_folder'], e['parent']['mtime'], e['parent']['id'])))
    return selected


def discover_lineage_edges(documents):
    result = []
    times = [document['mtime'] for document in documents]
    chronological = all(a <= b for a, b in zip(times, times[1:]))
    for new_index, new in enumerate(documents):
        candidates, fallback = [], []
        start_index = bisect_left(times, new['mtime'] - timedelta(days=MAX_PARENT_DAYS), 0, new_index) if chronological else 0
        for old_index in range(start_index, new_index):
            old = documents[old_index]
            if new['mtime'] - old['mtime'] > timedelta(days=MAX_PARENT_DAYS):
                continue
            similarity = compare_documents(old, new)
            edge = {'parent': old, 'child': new, **similarity}
            if ENHANCED_LINEAGE:
                edge.update(enhanced_pair_evidence(old, new))
            if similarity['score'] >= MIN_EDGE_SCORE:
                candidates.append(edge)
            elif (ENHANCED_LINEAGE and old['word_count'] and new['word_count']
                  and (edge['name_candidate'] or edge['shingle_candidate'])):
                fallback.append(edge)
        if ENHANCED_LINEAGE:
            result.extend(select_enhanced_parents(new, candidates, fallback))
        elif candidates:
            # Preserve original ordering, thresholds and parent cap in legacy mode.
            candidates.sort(key=lambda e: e['score'], reverse=True)
            best = candidates[0]['score']
            result.extend([e for e in candidates if e['score'] >= best-PARENT_SCORE_MARGIN][:MAX_PARENTS])
    return result


ENHANCED_EDGE_COLUMNS = {
    'Evidence_Type': 'evidence_type', 'Candidate_Score': 'candidate_score',
    'Candidate_Reason': 'candidate_reason', 'Uncertain_Lineage': 'uncertain',
    'Filename_Family_Match': 'filename_family_match', 'Version_Progression': 'version_progression',
    'Filename_Date_Order': 'filename_date_order', 'Same_Folder': 'same_folder',
    'Chronology_Evidence': 'chronology_evidence', 'Old_Shingle_Containment': 'old_shingle_containment',
    'New_Shingle_Containment': 'new_shingle_containment', 'Common_Shingles': 'common_shingles',
}


# ============================================================
# REUSABLE TEXT SNAPSHOT (no lineage decisions stored here)
# ============================================================

def text_cache_document(document):
    """Cache offsets use normalized words, 0-based with exclusive end offsets."""
    status = document['extraction_status']
    error = document['extraction_error']
    try:
        current = os.stat(document['path'])
        if (current.st_size, current.st_mtime_ns) != (document['size'], document['mtime_ns']):
            status, error = 'stale', 'File changed during collection; rescan before passage search.'
    except OSError as exc:
        status, error = 'unavailable', str(exc)
    if document['binary_hash'] == 'ERROR':
        status, error = 'error', error or 'File fingerprint could not be read.'
    paragraphs, offset = [], 0
    # Failed or unstable reads must not become usable matching evidence.
    if status == 'ok':
        for record in document['paragraph_records']:
            words = normalize_text(record['text']).split()
            paragraphs.append({**record, 'normalized_words': words,
                               'word_start': offset, 'word_end': offset + len(words)})
            offset += len(words)
    return {
        'id': document['id'], 'path': os.path.abspath(document['path']),
        'filename': document['filename'],
        'fingerprint': {'size_bytes': document['size'], 'mtime_ns': document['mtime_ns'],
                        'binary_sha256': document['binary_hash']},
        'normalized_text_sha256': document['content_hash'],
        'status': status, 'error': error, 'word_count': offset,
        'paragraphs': paragraphs,
    }


def write_text_cache(documents, destination):
    """Atomic compressed JSON; readers must check versions and file fingerprints.

    IDs are scoped to this scan. A future reader must match IDs AND full paths
    and hashes against the loaded nodes CSV, then check current size/mtime_ns.
    Headers, footers, footnotes, comments, formatting and images are excluded.
    """
    payload = {
        'schema_version': TEXT_CACHE_SCHEMA_VERSION,
        'extraction_version': TEXT_EXTRACTION_VERSION,
        'normalization_version': TEXT_NORMALIZATION_VERSION,
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'run_name': RUN_NAME, 'root_dir': os.path.abspath(ROOT_DIR),
        'scan_days_ago': DAYS_AGO,
        'nodes_file': os.path.abspath(NODES_FILE),
        'nodes_sha256': full_file_hash(NODES_FILE),
        'location_convention': '1-based XML paragraph ordinal; 0-based half-open normalized word spans',
        'documents': [text_cache_document(d) for d in documents],
    }
    fd, temporary = tempfile.mkstemp(prefix='.docx-text-cache-', suffix='.tmp',
                                     dir=os.path.dirname(os.path.abspath(destination)))
    os.close(fd)
    try:
        with gzip.open(temporary, 'wt', encoding='utf-8', compresslevel=3) as stream:
            json.dump(payload, stream, ensure_ascii=False, separators=(',', ':'))
        os.replace(temporary, destination)
    finally:
        if os.path.exists(temporary):
            os.remove(temporary)
    return payload


# ============================================================
# SCAN
# ============================================================

_builder_started = perf_counter()
cutoff = datetime.now() - timedelta(days=DAYS_AGO)

documents = []

print("Scanning DOCX files...")

for root, dirs, files in os.walk(ROOT_DIR):

    for filename in files:

        if not filename.lower().endswith(".docx"):
            continue

        if filename.startswith("~$"):
            continue

        path = os.path.join(root, filename)

        try:
            file_stat = os.stat(path)
            mtime = datetime.fromtimestamp(file_stat.st_mtime)
            ctime = datetime.fromtimestamp(file_stat.st_ctime)

        except OSError:
            continue

        if mtime < cutoff:
            continue

        print("Reading:", path)

        info = inspect_docx(path)
        info["filename_evidence"] = filename_evidence(filename)

        documents.append({
            "id": "",
            "filename": filename,
            "path": path,
            "folder": root,
            "mtime": mtime,
            "ctime": ctime,
            "size": file_stat.st_size,
            "mtime_ns": file_stat.st_mtime_ns,
            "binary_hash": full_file_hash(path),

            **info
        })


documents.sort(key=lambda d: d["mtime"])

for i, document in enumerate(documents):
    document["id"] = f"D{i+1:04d}"


print()
print("Documents found:", len(documents))


# ============================================================
# DISCOVER RELATIONSHIPS
# ============================================================

_scan_finished = perf_counter()
print(f"Scan/extraction completed in {_scan_finished - _builder_started:.1f}s. Discovering lineage...")
candidate_edges = discover_lineage_edges(documents)
_discovery_finished = perf_counter()
print(f"Lineage discovery completed in {_discovery_finished - _scan_finished:.1f}s.")


print()
print(
    "Candidate lineage edges:",
    len(candidate_edges)
)


# ============================================================
# NEW: DETAILED DIFF PASS
# ============================================================

detailed_count = 0

print()
print(
    "Running detailed comparison on "
    "high-similarity candidate pairs..."
)

for edge in candidate_edges:
    if edge.get('detail_diff_run'):
        detailed_count += 1  # Fallback verification already performed this diff.
        continue
    if should_run_detailed_diff(edge):
        edge.update(detailed_text_diff(edge['parent'], edge['child']))
        detailed_count += 1
    elif ENHANCED_LINEAGE and edge['score'] >= ENHANCED_DETAIL_THRESHOLD:
        detail = bounded_detail(edge)
        edge.update(detail if detail is not None else blank_detailed_diff())
        detailed_count += int(detail is not None)
    else:
        edge.update(blank_detailed_diff())


print(
    f"Detailed diff completed for "
    f"{detailed_count} of "
    f"{len(candidate_edges)} edges."
)


# ============================================================
# NODE CSV
# ============================================================

with open(
    NODES_FILE,
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    fields = [
        "ID",
        "Modified",
        "Windows_Created",
        "Filename",
        "Folder",
        "Size_Bytes",
        "Words",
        "Binary_SHA256",
        "Text_SHA256",
        "Word_Revision",
        "Core_Created",
        "Core_Modified",
        "Creator",
        "Last_Modified_By",
        "Tracked_Changes",
        "Comments",
        "Full_Path",
        "Filename_Family", "Filename_Versions", "Filename_Date", "Lineage_Mode",
    ]

    writer = csv.DictWriter(
        f,
        fieldnames=fields
    )

    writer.writeheader()

    for d in documents:

        metadata = d["metadata"]

        writer.writerow({
            "ID": d["id"],

            "Modified": d[
                "mtime"
            ].strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            "Windows_Created": d[
                "ctime"
            ].strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            "Filename": d["filename"],
            "Folder": d["folder"],
            "Size_Bytes": d["size"],
            "Words": d["word_count"],
            "Binary_SHA256": d["binary_hash"],
            "Text_SHA256": d["content_hash"],

            "Word_Revision": metadata.get(
                "revision", ""
            ),

            "Core_Created": metadata.get(
                "created", ""
            ),

            "Core_Modified": metadata.get(
                "modified", ""
            ),

            "Creator": metadata.get(
                "creator", ""
            ),

            "Last_Modified_By": metadata.get(
                "last_modified_by", ""
            ),

            "Tracked_Changes": d[
                "tracked_changes"
            ],

            "Comments": d["comments"],
            "Full_Path": d["path"],
            "Filename_Family": d["filename_evidence"]["family"],
            "Filename_Versions": "_".join('v'+str(v) for v in d["filename_evidence"]["versions"]),
            "Filename_Date": d["filename_evidence"]["date"].strftime('%Y-%m-%d') if d["filename_evidence"]["date"] else '',
            "Lineage_Mode": "enhanced" if ENHANCED_LINEAGE else "legacy",
        })


# ============================================================
# EDGE CSV
# ============================================================

with open(
    EDGES_FILE,
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    fields = [
        "Parent_ID",
        "Parent_File",
        "Child_ID",
        "Child_File",
        "Relationship",
        "Score",
        "Old_Content_In_New",
        "New_Content_In_Old",
        "Paragraph_Jaccard",
        "Shingle_Jaccard",
        "Common_Paragraphs",

        "Detail_Diff_Run",
        "Exact_Visible_Text_Equal",
        "Diff_Class",

        "Paragraph_Edit_Blocks",
        "Changed_Paragraphs_Est",
        "Paragraphs_Inserted",
        "Paragraphs_Deleted",
        "Paragraphs_Replaced_Old",
        "Paragraphs_Replaced_New",

        "Diff_Edit_Blocks",
        "Inserted_Tokens",
        "Deleted_Tokens",
        "Replaced_Old_Tokens",
        "Replaced_New_Tokens",
        "Changed_Tokens_Est",

        "Token_Similarity",
        "Text_Delta_Pct",
        "Old_Token_Count",
        "New_Token_Count",

        "Changed_Old_Sample",
        "Changed_New_Sample",

        "Parent_Modified",
        "Child_Modified",
        "Time_Gap_Hours",
        *ENHANCED_EDGE_COLUMNS,
    ]

    writer = csv.DictWriter(
        f,
        fieldnames=fields
    )

    writer.writeheader()

    for edge in candidate_edges:

        time_gap_hours = (
            edge["child"]["mtime"]
            - edge["parent"]["mtime"]
        ).total_seconds() / 3600.0

        token_similarity = edge.get(
            "token_similarity", ""
        )

        text_delta_pct = edge.get(
            "text_delta_pct", ""
        )

        writer.writerow({
            **{column: edge.get(key, 'content' if column == 'Evidence_Type' else '')
               for column, key in ENHANCED_EDGE_COLUMNS.items()},
            "Parent_ID": edge[
                "parent"
            ]["id"],

            "Parent_File": edge[
                "parent"
            ]["filename"],

            "Child_ID": edge[
                "child"
            ]["id"],

            "Child_File": edge[
                "child"
            ]["filename"],

            "Relationship": edge[
                "relationship"
            ],

            "Score": round(
                edge["score"], 4
            ),

            "Old_Content_In_New": round(
                edge["old_in_new"], 4
            ),

            "New_Content_In_Old": round(
                edge["new_in_old"], 4
            ),

            "Paragraph_Jaccard": round(
                edge["paragraph_jaccard"], 4
            ),

            "Shingle_Jaccard": round(
                edge["shingle_jaccard"], 4
            ),

            "Common_Paragraphs": edge[
                "common_paragraphs"
            ],

            "Detail_Diff_Run": edge.get(
                "detail_diff_run", False
            ),

            "Exact_Visible_Text_Equal": edge.get(
                "exact_visible_text_equal", ""
            ),

            "Diff_Class": edge.get(
                "diff_class", ""
            ),

            "Paragraph_Edit_Blocks": edge.get(
                "paragraph_edit_blocks", ""
            ),

            "Changed_Paragraphs_Est": edge.get(
                "changed_paragraphs_est", ""
            ),

            "Paragraphs_Inserted": edge.get(
                "paragraphs_inserted", ""
            ),

            "Paragraphs_Deleted": edge.get(
                "paragraphs_deleted", ""
            ),

            "Paragraphs_Replaced_Old": edge.get(
                "paragraphs_replaced_old", ""
            ),

            "Paragraphs_Replaced_New": edge.get(
                "paragraphs_replaced_new", ""
            ),

            "Diff_Edit_Blocks": edge.get(
                "diff_edit_blocks", ""
            ),

            "Inserted_Tokens": edge.get(
                "inserted_tokens", ""
            ),

            "Deleted_Tokens": edge.get(
                "deleted_tokens", ""
            ),

            "Replaced_Old_Tokens": edge.get(
                "replaced_old_tokens", ""
            ),

            "Replaced_New_Tokens": edge.get(
                "replaced_new_tokens", ""
            ),

            "Changed_Tokens_Est": edge.get(
                "changed_tokens_est", ""
            ),

            "Token_Similarity": (
                round(
                    token_similarity,
                    6
                )
                if token_similarity != ""
                else ""
            ),

            "Text_Delta_Pct": (
                round(
                    text_delta_pct,
                    4
                )
                if text_delta_pct != ""
                else ""
            ),

            "Old_Token_Count": edge.get(
                "old_token_count", ""
            ),

            "New_Token_Count": edge.get(
                "new_token_count", ""
            ),

            "Changed_Old_Sample": edge.get(
                "changed_old_sample", ""
            ),

            "Changed_New_Sample": edge.get(
                "changed_new_sample", ""
            ),

            "Parent_Modified": edge[
                "parent"
            ]["mtime"].strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            "Child_Modified": edge[
                "child"
            ]["mtime"].strftime(
                "%Y-%m-%d %H:%M:%S"
            ),

            "Time_Gap_Hours": round(
                time_gap_hours,
                4
            ),
        })


# ============================================================
# GRAPHVIZ DOT
# ============================================================

def dot_escape(value):
    return (
        str(value)
        .replace("\\", "\\\\")
        .replace('"', '\\"')
    )


with open(
    DOT_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "digraph DocumentHistory {\n"
    )
    f.write(
        '  rankdir="LR";\n'
    )
    f.write(
        '  node [shape=box, fontsize=9];\n\n'
    )

    for d in documents:

        label = (
            f'{d["id"]}\\n'
            f'{d["mtime"].strftime("%Y-%m-%d %H:%M")}\\n'
            f'{d["filename"]}\\n'
            f'{d["word_count"]:,} words'
        )

        f.write(
            f'  {d["id"]} '
            f'[label="{dot_escape(label)}"];\n'
        )

    f.write("\n")

    for edge in candidate_edges:

        parent = edge["parent"]["id"]
        child = edge["child"]["id"]

        label_lines = [
            edge["relationship"],
            f'{edge["score"]:.2f}'
        ]

        if edge.get('uncertain'):
            label_lines.insert(0, 'CANDIDATE: ' + edge.get('evidence_type', ''))
            label_lines.append('candidate rank ' + str(edge.get('candidate_score', '')))
        if edge.get(
            "detail_diff_run",
            False
        ):

            label_lines.append(
                f'Δ {edge["text_delta_pct"]:.2f}%'
            )

            label_lines.append(
                f'{edge["diff_edit_blocks"]} edit blocks'
            )

        label = "\\n".join(
            label_lines
        )

        candidate_style = ', style="dashed", color="#806b91"' if edge.get('uncertain') else ''
        if edge.get('chronology_evidence') == 'conflict':
            candidate_style = ', style="dashed", color="#cc4c4c"'
        f.write(
            f'  {parent} -> {child} '
            f'[label="{dot_escape(label)}"{candidate_style}];\n'
        )

    f.write("}\n")


if WRITE_TEXT_CACHE:
    _cache_started = perf_counter()
    _text_snapshot = write_text_cache(documents, TEXT_CACHE_FILE)
    _cache_usable = sum(d['status'] == 'ok' for d in _text_snapshot['documents'])
    print(f"Text cache: {TEXT_CACHE_FILE}")
    print(f"  {_cache_usable}/{len(documents)} usable documents; "
          f"{perf_counter() - _cache_started:.2f}s to build/write cache.")
    del _text_snapshot


# ============================================================
# FINISH
# ============================================================

print()
print("=" * 70)
print("Finished.")
print()
print("Search folder:")
print(f"  {ROOT_DIR}")
print()
print("Created:")
print(f"  Nodes : {NODES_FILE}")
print(f"  Edges : {EDGES_FILE}")
print(f"  Graph : {DOT_FILE}")
print()
print(
    f"Detailed pairwise diff was run on "
    f"{detailed_count} high-similarity edges."
)
print()
print("Important new edge columns:")
print("  Detail_Diff_Run")
print("  Exact_Visible_Text_Equal")
print("  Diff_Class")
print("  Paragraph_Edit_Blocks")
print("  Changed_Paragraphs_Est")
print("  Diff_Edit_Blocks")
print("  Inserted_Tokens / Deleted_Tokens")
print("  Replaced_Old_Tokens / Replaced_New_Tokens")
print("  Token_Similarity")
print("  Text_Delta_Pct")
print("  Changed_Old_Sample / Changed_New_Sample")
print("=" * 70)

print("Lineage mode:", "enhanced" if ENHANCED_LINEAGE else "legacy")
print("Uncertain candidate edges:", sum(bool(e.get("uncertain")) for e in candidate_edges))

print(f"Total builder time: {perf_counter() - _builder_started:.1f}s")
print("Reused pair calculations:", _builder_cache_hits)
print("Tokenization cache:", tokenize_for_diff.cache_info())


# The third version: Run this if a scan of previously created .docx files has already generated _edges.csv, _nodes.csv and _text_cache.json.gz. The dashboard will then display the file history/lineage for the selected file. Open the dashboard by entering 127.0.0.1:8050 into your browser. 

Optional text-block search: expand **Search source files by matching text blocks**, select a target, and click **Find text contributors**. Requires the matching text cache from the first 3rd cell. Default: at least 24 matching words per block and 85% local word agreement, anchored by exact normalized five-word sequences. Results include exact/lightly edited wording, paragraph locations and side-by-side evidence; graph overlays come in a later step.


Step 3: the passage report selects complementary contributor groups, initially showing six with an expandable remainder. Files matching substantially the same target positions are alternatives, not automatically a version lineage. Near-whole-document and multi-section matches are retained separately as possible assembled versions; common wording and redundant matches remain accessible for review. Coverage uses representative files only.


Step 4: after a target search, green dashed text-contribution arrows and contributor ancestor branches appear in the graph. Toggle the overlay, show all selected groups, or include alternative files in the search controls. Click a contribution arrow for matching passages in its physical source and target. History branches follow existing graph filters; contribution arrows are retained independently. Changing target clears the search state.

Intermediate assemblies and specific matching files can now be included independently of compact component selection. Modification timestamps can be displayed in both layouts; grouped nodes show their earliest copy timestamp.


In [ ]:
# 3rd version : Dash Lineage" uses csv file  nodes and edges from cell with 3rd
# http://127.0.0.1:8050/
import os
import json
import gzip
import uuid
from threading import Lock
import re
import hashlib
import textwrap
import difflib
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, deque, OrderedDict
from functools import lru_cache
import pandas as pd
import dash
from dash import html, dcc, Input, Output, State, ALL
import dash_cytoscape as cyto

ROOT_DIR = r"Z:\TB"
OUTPUT_DIR = ROOT_DIR
RUN_NAME = "Working_File2nd"

NODES_FILE_NAME = f"{RUN_NAME}_nodes.csv"
EDGES_FILE_NAME = f"{RUN_NAME}_edges.csv"
DOT_FILE_NAME   = f"{RUN_NAME}_history.dot"

NODES_FILE = os.path.join(OUTPUT_DIR, NODES_FILE_NAME)#'Z:\\TB\\TEXT-FIGURE formulation Preparation-202604-\\Working_File\\Working_File_nodes.csv'
EDGES_FILE = os.path.join(OUTPUT_DIR, EDGES_FILE_NAME)#'Z:\\TB\\TEXT-FIGURE formulation Preparation-202604-\\Working_File\\Working_File_edges.csv'

# NODES_FILE_NAME = f"{RUN_NAME}_nodes.csv"
# EDGES_FILE_NAME = f"{RUN_NAME}_edges.csv"
# http://127.0.0.1:8050/
PORT = 8050
DEFAULT_DIRECTION = 'ancestors'
DEFAULT_DEPTH = 3
DEFAULT_MIN_SCORE = 0.55
DEFAULT_LAYOUT = 'time'
DEFAULT_VIEW_MODE = 'likely'
DEFAULT_PARENTS_PER_NODE = 2
DEFAULT_COLLAPSE_MODE = 'binary'
# Display settings only; CSV evidence and predecessor ranking are unchanged.
NODE_FONT_SIZE = 12
SELECTED_FONT_SIZE = 14
EDGE_FONT_SIZE = 10
LABEL_MAX_WIDTH = 240
TREE_SPACING_FACTOR = 1.2
TIME_HORIZONTAL_SPAN = 1100
TIME_COMPACT_STEP = 280
DEFAULT_TIME_SPACING = 'compact'
TIME_GENERATION_GAP = 150
TIME_LABEL_GAP = 24
DEFAULT_BRANCH_COLORS = True
BRANCH_PALETTE = ('#83b6a3', '#a99bc9', '#cfb581', '#8cabc9', '#c59cae', '#9bb888')
NEUTRAL_NODE_COLOR = '#7aa6c2'
TIME_TOLERANCE_HOURS = 3.0
WEIGHT_CONTENT = 0.55
WEIGHT_TIME = 0.25
WEIGHT_WORDS = 0.15
WEIGHT_RELATIONSHIP = 0.05
nodes = pd.read_csv(NODES_FILE)
edges = pd.read_csv(EDGES_FILE)
nodes['ID'] = nodes['ID'].astype(str)
edges['Parent_ID'] = edges['Parent_ID'].astype(str)
edges['Child_ID'] = edges['Child_ID'].astype(str)
numeric_edge_cols = ['Score', 'Old_Content_In_New', 'New_Content_In_Old', 'Paragraph_Jaccard', 'Shingle_Jaccard', 'Common_Paragraphs', 'Paragraph_Edit_Blocks', 'Changed_Paragraphs_Est', 'Paragraphs_Inserted', 'Paragraphs_Deleted', 'Paragraphs_Replaced_Old', 'Paragraphs_Replaced_New', 'Diff_Edit_Blocks', 'Inserted_Tokens', 'Deleted_Tokens', 'Replaced_Old_Tokens', 'Replaced_New_Tokens', 'Changed_Tokens_Est', 'Token_Similarity', 'Text_Delta_Pct', 'Old_Token_Count', 'New_Token_Count', 'Time_Gap_Hours']
for col in numeric_edge_cols:
    if col in edges.columns:
        edges[col] = pd.to_numeric(edges[col], errors='coerce')
if 'Score' not in edges.columns:
    edges['Score'] = 0.0
edges['Score'] = edges['Score'].fillna(0.0)
nodes['_Modified_DT'] = pd.to_datetime(nodes['Modified'], errors='coerce') if 'Modified' in nodes.columns else pd.NaT
nodes['_Words_Num'] = pd.to_numeric(nodes['Words'], errors='coerce') if 'Words' in nodes.columns else pd.NA
print(f'Loaded {len(nodes)} documents and {len(edges)} relationships.')

def safe_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default

def safe_int(value, default=0):
    try:
        if pd.isna(value):
            return default
        return int(round(float(value)))
    except Exception:
        return default

def clean_text(value):
    if value is None or pd.isna(value):
        return ''
    return str(value)

def clean_hash(value):
    value = clean_text(value).strip()
    if not value or value.lower() in {'nan', 'none', 'error'}:
        return ''
    return value

def as_bool(value):
    if isinstance(value, bool):
        return value
    text = clean_text(value).strip().lower()
    return text in {'true', '1', 'yes', 'y'}

def format_gap(hours):
    if hours is None:
        return 'unknown'
    sign = '-' if hours < 0 else ''
    total_minutes = int(round(abs(hours) * 60))
    days, rem = divmod(total_minutes, 1440)
    hrs, mins = divmod(rem, 60)
    if days:
        return f'{sign}{days}d {hrs}h {mins}m'
    if hrs:
        return f'{sign}{hrs}h {mins}m'
    return f'{sign}{mins}m'

def relationship_bonus(relationship):
    text = clean_text(relationship).lower()
    if 'same text' in text or 'renamed copy' in text:
        return 1.0
    if 'revision' in text:
        return 0.95
    if 'expanded' in text or 'copied into' in text:
        return 0.75
    if 'partial reuse' in text:
        return 0.45
    if 'excerpt' in text or 'extracted' in text:
        return 0.3
    return 0.5

def diff_summary_from_row(row):
    if not as_bool(row.get('Detail_Diff_Run', False)):
        return 'Diff not run'
    if as_bool(row.get('Exact_Visible_Text_Equal', False)):
        return 'Same extracted text'
    delta = safe_float(row.get('Text_Delta_Pct'), None)
    changed = safe_float(row.get('Changed_Tokens_Est'), 0)
    denominator = max(safe_float(row.get('Old_Token_Count'), 0),
                      safe_float(row.get('New_Token_Count'), 0))
    # Recover tiny changes rounded in the CSV from the exported exact counts.
    if changed > 0 and denominator > 0:
        delta = 100.0 * changed / denominator
    parts = []
    if delta is not None:
        if delta > 0:
            value = f'{delta:.4f}'.rstrip('0').rstrip('.') if delta >= 0.0001 else '<0.0001'
            parts.append('Δ' + value + '%')
        elif changed > 0 or safe_int(row.get('Diff_Edit_Blocks')) > 0:
            parts.append('Text changed; Δ below saved precision')
        else:
            parts.append('Δ0% tokens')
    for column, label in (('Diff_Edit_Blocks', 'edits'), ('Changed_Paragraphs_Est', 'paras')):
        value = safe_float(row.get(column), None)
        if value is not None:
            parts.append(f'{int(value)} {label}')
    return ' | '.join(parts) or 'Diff details unavailable'


def is_copy_edge(row, graph=None):
    if as_bool(row.get('Detail_Diff_Run', False)):
        return as_bool(row.get('Exact_Visible_Text_Equal', False))
    # Older exports may lack detailed diffs; only a valid binary match proves
    # a copy in that case. A normalized-text match can still contain small edits.
    if graph is not None:
        a = graph['nodes'].get(str(row.get('Parent_ID')), {})
        b = graph['nodes'].get(str(row.get('Child_ID')), {})
        digest = clean_hash(a.get('binary_hash', ''))
        return bool(digest and digest == clean_hash(b.get('binary_hash', '')))
    return False

def duplicate_key(row, collapse_mode):
    if collapse_mode == 'off':
        return 'NODE:' + str(row['ID'])
    if collapse_mode == 'binary':
        binary_hash = clean_hash(row.get('Binary_SHA256', ''))
        return 'BIN:' + binary_hash if binary_hash else 'NODE:' + str(row['ID'])
    text_hash = clean_hash(row.get('Text_SHA256', ''))
    if text_hash:
        return 'TEXT:' + text_hash
    binary_hash = clean_hash(row.get('Binary_SHA256', ''))
    if binary_hash:
        return 'BIN:' + binary_hash
    return 'NODE:' + str(row['ID'])

def is_candidate_edge(row):
    return clean_text(row.get('Evidence_Type', '')) in {'filename_candidate', 'shingle_candidate'}


def best_edge_row(rows_for_pair):
    """Prefer strongest edge; on ties prefer a row with detailed diff."""

    def keyfunc(r):
        return (not is_candidate_edge(r), safe_float(r.get('Score', 0.0)), 1 if as_bool(r.get('Detail_Diff_Run', False)) else 0, safe_float(r.get('Token_Similarity', 0.0)))
    return max(rows_for_pair, key=keyfunc)

# Caches belong to this loaded CSV snapshot; rerunning the cell creates fresh caches.
@lru_cache(maxsize=3)
def build_graph_data(collapse_mode='binary'):
    work = nodes.copy()
    work['_Group_Key'] = work.apply(lambda row: duplicate_key(row, collapse_mode), axis=1)
    grouped_rows = {key: group.copy() for key, group in work.groupby('_Group_Key', sort=False)}
    original_to_group = {}
    group_nodes = {}
    for group_key, group in grouped_rows.items():
        member_ids = sorted(group['ID'].astype(str).tolist())
        logical_id = member_ids[0]
        for member_id in member_ids:
            original_to_group[member_id] = logical_id
        valid_times = group['_Modified_DT'].dropna()
        earliest_dt = valid_times.min() if not valid_times.empty else pd.NaT
        latest_dt = valid_times.max() if not valid_times.empty else pd.NaT
        if not valid_times.empty:
            representative_idx = group['_Modified_DT'].idxmin()
            representative = group.loc[representative_idx]
        else:
            representative = group.iloc[0]
        paths = [str(x) for x in group.get('Full_Path', pd.Series(dtype=str)).tolist() if not pd.isna(x)]
        filenames = [str(x) for x in group.get('Filename', pd.Series(dtype=str)).tolist() if not pd.isna(x)]
        revisions = [str(x) for x in group.get('Word_Revision', pd.Series(dtype=str)).tolist() if not pd.isna(x)]
        creators = [str(x) for x in group.get('Creator', pd.Series(dtype=str)).tolist() if not pd.isna(x)]
        unique_filenames = list(dict.fromkeys(filenames))
        unique_revisions = list(dict.fromkeys(revisions))
        unique_creators = list(dict.fromkeys(creators))
        base_filename = str(representative.get('Filename', logical_id))
        display_filename = base_filename
        if len(member_ids) > 1:
            display_filename = f'{base_filename}  [{len(member_ids)} copies]'
        group_nodes[logical_id] = {'id': logical_id, 'member_ids': member_ids, 'copy_count': len(member_ids), 'filename': display_filename, 'representative_filename': base_filename, 'filenames': unique_filenames, 'earliest_dt': earliest_dt, 'latest_dt': latest_dt, 'modified': '' if pd.isna(earliest_dt) else pd.Timestamp(earliest_dt).strftime('%Y-%m-%d %H:%M:%S'), 'modified_latest': '' if pd.isna(latest_dt) else pd.Timestamp(latest_dt).strftime('%Y-%m-%d %H:%M:%S'), 'words': representative.get('Words', ''), 'words_num': None if pd.isna(representative.get('_Words_Num', pd.NA)) else float(representative.get('_Words_Num')), 'revision': ', '.join(unique_revisions), 'creator': ', '.join(unique_creators), 'paths': paths, 'path': paths[0] if paths else '', 'text_hash': clean_hash(representative.get('Text_SHA256', '')), 'binary_hash': clean_hash(representative.get('Binary_SHA256', ''))}
    pair_to_rows = defaultdict(list)
    for _, row in edges.iterrows():
        raw_source = str(row['Parent_ID'])
        raw_target = str(row['Child_ID'])
        source = original_to_group.get(raw_source, raw_source)
        target = original_to_group.get(raw_target, raw_target)
        if source == target:
            continue
        pair_to_rows[source, target].append(row)
    grouped_edges = []
    detail_cols = ['Evidence_Type', 'Candidate_Score', 'Candidate_Reason', 'Uncertain_Lineage', 'Filename_Family_Match', 'Version_Progression', 'Filename_Date_Order', 'Same_Folder', 'Chronology_Evidence', 'Old_Shingle_Containment', 'New_Shingle_Containment', 'Common_Shingles'] + ['Detail_Diff_Run', 'Exact_Visible_Text_Equal', 'Diff_Class', 'Paragraph_Edit_Blocks', 'Changed_Paragraphs_Est', 'Paragraphs_Inserted', 'Paragraphs_Deleted', 'Paragraphs_Replaced_Old', 'Paragraphs_Replaced_New', 'Diff_Edit_Blocks', 'Inserted_Tokens', 'Deleted_Tokens', 'Replaced_Old_Tokens', 'Replaced_New_Tokens', 'Changed_Tokens_Est', 'Token_Similarity', 'Text_Delta_Pct', 'Old_Token_Count', 'New_Token_Count', 'Changed_Old_Sample', 'Changed_New_Sample']
    for (source, target), rows_for_pair in pair_to_rows.items():
        best = best_edge_row(rows_for_pair)
        relationships = sorted({clean_text(r.get('Relationship', '')) for r in rows_for_pair if clean_text(r.get('Relationship', '')).strip()})
        item = {'Parent_ID': source, 'Child_ID': target, 'Score': max((safe_float(r.get('Score', 0.0)) for r in rows_for_pair)), 'Relationship': ' / '.join(relationships), 'Old_Content_In_New': safe_float(best.get('Old_Content_In_New', 0.0)), 'New_Content_In_Old': safe_float(best.get('New_Content_In_Old', 0.0)), 'Paragraph_Jaccard': safe_float(best.get('Paragraph_Jaccard', 0.0)), 'Shingle_Jaccard': safe_float(best.get('Shingle_Jaccard', 0.0)), 'Common_Paragraphs': safe_float(best.get('Common_Paragraphs', 0.0)), 'Evidence_Edge_Count': len(rows_for_pair)}
        item['Evidence_Parent_ID'] = str(best['Parent_ID'])
        item['Evidence_Child_ID'] = str(best['Child_ID'])
        for col in detail_cols:
            item[col] = best.get(col, '')
        grouped_edges.append(item)
    grouped_edges = pd.DataFrame(grouped_edges)
    if grouped_edges.empty:
        grouped_edges = pd.DataFrame(columns=['Parent_ID', 'Child_ID', 'Score', 'Relationship', 'Old_Content_In_New', 'New_Content_In_Old', 'Paragraph_Jaccard', 'Shingle_Jaccard', 'Common_Paragraphs', 'Evidence_Edge_Count', *detail_cols])
    children, parents = make_adjacency(grouped_edges)
    return {'nodes': group_nodes, 'edges': grouped_edges, 'children': children, 'parents': parents, 'original_to_group': original_to_group}

def make_adjacency(edge_df):
    children = defaultdict(list)
    parents = defaultdict(list)
    for _, row in edge_df.iterrows():
        parent = str(row['Parent_ID'])
        child = str(row['Child_ID'])
        score = safe_float(row.get('Score', 0.0))
        relationship = clean_text(row.get('Relationship', ''))
        children[parent].append((child, score, relationship))
        parents[child].append((parent, score, relationship))
    return (children, parents)

def get_group_dt(graph, node_id):
    value = graph['nodes'].get(node_id, {}).get('earliest_dt', pd.NaT)
    if pd.isna(value):
        return None
    return pd.Timestamp(value).to_pydatetime()

def get_group_words(graph, node_id):
    value = graph['nodes'].get(node_id, {}).get('words_num', None)
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return None

def temporal_score(graph, parent_id, child_id):
    pdt = get_group_dt(graph, parent_id)
    cdt = get_group_dt(graph, child_id)
    if pdt is None or cdt is None:
        return (0.5, None, 'unknown')
    hours = (cdt - pdt).total_seconds() / 3600.0
    if hours < -TIME_TOLERANCE_HOURS:
        return (0.05, hours, 'conflict')
    if hours < 0:
        score = max(0.55, 1.0 - abs(hours) / max(TIME_TOLERANCE_HOURS, 0.1) * 0.35)
        return (score, hours, 'ambiguous/autosave')
    days = hours / 24.0
    score = 1.0 / (1.0 + days / 7.0)
    if hours <= 6:
        status = 'very close'
    elif hours <= 24:
        status = 'same day'
    elif days <= 7:
        status = 'plausible'
    else:
        status = 'older'
    return (score, hours, status)

def word_similarity(graph, parent_id, child_id):
    pw = get_group_words(graph, parent_id)
    cw = get_group_words(graph, child_id)
    if not pw or not cw or max(pw, cw) <= 0:
        return (0.5, None)
    similarity = min(pw, cw) / max(pw, cw)
    delta_pct = (cw - pw) / pw * 100.0 if pw else None
    return (similarity, delta_pct)

def predecessor_rank(graph, parent_id, child_id, edge_row):
    content = safe_float(edge_row.get('Score', 0.0))
    tscore, gap_hours, chronology = temporal_score(graph, parent_id, child_id)
    wscore, word_delta_pct = word_similarity(graph, parent_id, child_id)
    rscore = relationship_bonus(edge_row.get('Relationship', ''))
    combined = WEIGHT_CONTENT * content + WEIGHT_TIME * tscore + WEIGHT_WORDS * wscore + WEIGHT_RELATIONSHIP * rscore
    if is_candidate_edge(edge_row):
        combined = safe_float(edge_row.get('Candidate_Score', 0.0))
    return {'combined': combined, 'content': content, 'time_score': tscore, 'word_similarity': wscore, 'relationship_score': rscore, 'gap_hours': gap_hours, 'chronology': chronology, 'word_delta_pct': word_delta_pct}

def has_alternative_path(edge_df, source, target, skip_index):
    adjacency = defaultdict(list)
    for idx, row in edge_df.iterrows():
        if idx == skip_index:
            continue
        adjacency[str(row['Parent_ID'])].append(str(row['Child_ID']))
    queue = deque([source])
    seen = {source}
    while queue:
        current = queue.popleft()
        for nxt in adjacency.get(current, []):
            if nxt == target:
                return True
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return False

def remove_transitive_edges(edge_df):
    if len(edge_df) < 3:
        return edge_df.copy()
    # Build once, retain edge indices so skipped/parallel edges behave exactly
    # as in the original per-edge search (including cyclic evidence graphs).
    adjacency = defaultdict(list)
    edge_rows = list(zip(edge_df.index, edge_df['Parent_ID'].astype(str), edge_df['Child_ID'].astype(str)))
    for idx, source, target in edge_rows:
        adjacency[source].append((target, idx))
    remove_indices = []
    for skip_index, source, target in edge_rows:
        queue = deque([source])
        seen = {source}
        found = False
        while queue and not found:
            current = queue.popleft()
            for neighbor, edge_index in adjacency.get(current, ()):
                if edge_index == skip_index:
                    continue
                if neighbor == target:
                    found = True
                    break
                if neighbor not in seen:
                    seen.add(neighbor)
                    queue.append(neighbor)
        if found:
            remove_indices.append(skip_index)
    return edge_df.drop(index=remove_indices).reset_index(drop=True)


@lru_cache(maxsize=48)
def cached_view_graph(collapse_mode, view_mode, parents_per_node, min_score, hide_transitive, include_candidates):
    return prepare_view_graph(build_graph_data(collapse_mode), view_mode, parents_per_node,
                              min_score, hide_transitive, include_candidates)


def prepare_view_graph(base_graph, view_mode, parents_per_node, min_score, hide_transitive, include_candidates=True):
    edge_df = base_graph['edges'].copy()
    candidate_mask = edge_df.apply(is_candidate_edge, axis=1).astype(bool) if not edge_df.empty else pd.Series(False, index=edge_df.index, dtype=bool)
    edge_df = edge_df[((~candidate_mask) & (edge_df['Score'] >= min_score)) | (candidate_mask & include_candidates)].copy()
    if view_mode == 'likely' and (not edge_df.empty):
        ranked_rows = []
        for child_id, group in edge_df.groupby('Child_ID', sort=False):
            temp = group.copy()
            ranks = []
            for _, row in temp.iterrows():
                rank = predecessor_rank(base_graph, str(row['Parent_ID']), str(row['Child_ID']), row)
                ranks.append(rank['combined'])
            temp['_Pred_Rank'] = ranks
            temp['_Candidate'] = temp.apply(is_candidate_edge, axis=1)
            temp['_Copy'] = temp.apply(lambda row: is_copy_edge(row, base_graph), axis=1)
            temp = temp.sort_values(['_Candidate', '_Pred_Rank', 'Score'], ascending=[True, False, False])
            # Copies have their own allowance, matching the builder policy.
            temp = pd.concat([temp[~temp['_Copy']].head(int(parents_per_node)),
                              temp[temp['_Copy']].head(1)])
            ranked_rows.append(temp)
        edge_df = pd.concat(ranked_rows, ignore_index=True) if ranked_rows else edge_df.iloc[0:0].copy()
    if hide_transitive and (not edge_df.empty):
        inferred = edge_df.apply(is_candidate_edge, axis=1).astype(bool)
        edge_df = pd.concat([remove_transitive_edges(edge_df[~inferred]), edge_df[inferred]], ignore_index=True)
    children, parents = make_adjacency(edge_df)
    return {**base_graph, 'edges': edge_df, 'children': children, 'parents': parents}

def traverse(start, adjacency, max_depth):
    found = {start}
    queue = deque([(start, 0)])
    while queue:
        current, depth = queue.popleft()
        if depth >= max_depth:
            continue
        for neighbor, score, relationship in adjacency.get(current, []):
            if neighbor not in found:
                found.add(neighbor)
                queue.append((neighbor, depth + 1))
    return found

def selected_nodes(graph, selected_id, direction, depth):
    if not selected_id:
        return set()
    keep = {selected_id}
    if direction in ('both', 'ancestors'):
        keep |= traverse(selected_id, graph['parents'], depth)
    if direction in ('both', 'descendants'):
        keep |= traverse(selected_id, graph['children'], depth)
    return keep

def node_label(row, node_id, layout_mode, show_dates=None):
    filename = clean_text(row.get('filename', node_id))
    modified = clean_text(row.get('modified', ''))
    enabled = layout_mode == 'time' if show_dates is None else show_dates
    if enabled and modified:
        prefix = modified + (' (earliest copy)' if int(row.get('copy_count', 1)) > 1 else '')
        return f'{prefix}\n{filename}'
    return filename


def time_positions(graph, keep, selected_id, spacing='compact'):
    """Compact chronology by default, with optional proportional elapsed time.

    Reuse horizontal lanes across generations; move vertically only where label
    boxes would overlap in proportional mode. Compact mode gives each physical
    node its own horizontal slot, including equal-time copies.
    """
    if not keep:
        return {}
    dts = {nid: get_group_dt(graph, nid) for nid in keep}
    times = sorted({dt for dt in dts.values() if dt is not None})
    time_index = {dt: i for i, dt in enumerate(times)}
    span = max((times[-1] - times[0]).total_seconds(), 1.0) if times else 1.0
    horizontal_span = min(TIME_HORIZONTAL_SPAN, max(1, len(times)-1) * TIME_COMPACT_STEP)
    chars_per_line = max(1, int(LABEL_MAX_WIDTH / SELECTED_FONT_SIZE))
    positions = {}
    boxes = []
    ids = sorted(keep, key=lambda nid: (dts[nid] is None, dts[nid] or pd.Timestamp.max.to_pydatetime(), nid))
    for slot, nid in enumerate(ids):
        dt = dts[nid]
        if spacing == 'compact':
            x = 80 + slot * max(TIME_COMPACT_STEP, LABEL_MAX_WIDTH + TIME_LABEL_GAP)
        elif dt is None:
            x = 80 + (len(times) * TIME_COMPACT_STEP if spacing == 'compact' else horizontal_span + TIME_COMPACT_STEP)
        elif spacing == 'proportional':
            x = 80 + (dt-times[0]).total_seconds() / span * horizontal_span
        else:
            x = 80 + time_index[dt] * TIME_COMPACT_STEP
        lines = sum(max(1, len(textwrap.wrap(line, width=chars_per_line)))
                    for line in node_label(graph['nodes'].get(nid, {}), nid, 'time').split('\n'))
        # Symmetric conservative bounds cover the node and its label above it.
        half_height = max(45, lines * SELECTED_FONT_SIZE * 1.4 + 40)
        blockers = [box for box in boxes if abs(x-box[0]) < LABEL_MAX_WIDTH + TIME_LABEL_GAP]
        y = 350.0
        while any(abs(y-other_y) < half_height+other_half+TIME_LABEL_GAP
                  for _,other_y,other_half in blockers):
            y += TIME_GENERATION_GAP
        positions[nid] = {'x': x, 'y': y}
        boxes.append((x,y,half_height))
    return positions


def filename_family(filename):
    """Only strip a trailing version suffix; this is a name hint, not lineage."""
    stem = os.path.splitext(clean_text(filename))[0].casefold()
    return re.sub(r'(?:[_ -]+v\d+)+$', '', stem).strip()


def filename_relatives(graph, selected_id):
    selected = graph['nodes'].get(selected_id, {})
    families = {filename_family(name) for name in selected.get('filenames', [])}
    families.discard('')
    return {nid for nid,row in graph['nodes'].items()
            if families.intersection(filename_family(name) for name in row.get('filenames', []))}


def branch_colors(graph, keep):
    """Color displayed splits; merges and unresolved cycles stay neutral.

    Single-parent continuations inherit their branch. Colors are visual guides,
    not evidence of authorship or save order; the palette may repeat.
    """
    children = {nid: set() for nid in keep}
    parents = {nid: set() for nid in keep}
    for nid in keep:
        for child, _, _ in graph['children'].get(nid, []):
            if child in keep:
                children[nid].add(child)
                parents[child].add(nid)
    remaining = {nid: len(parents[nid]) for nid in keep}
    queue = deque(sorted(nid for nid in keep if remaining[nid] == 0))
    colors = dict.fromkeys(keep, NEUTRAL_NODE_COLOR)
    while queue:
        nid = queue.popleft()
        if len(parents[nid]) == 1:
            parent = next(iter(parents[nid]))
            if len(children[parent]) > 1:
                siblings = sorted(children[parent])
                offset = int(hashlib.sha256(parent.encode('utf-8')).hexdigest()[:8], 16)
                colors[nid] = BRANCH_PALETTE[(offset + siblings.index(nid)) % len(BRANCH_PALETTE)]
            else:
                colors[nid] = colors[parent]
        for child in sorted(children[nid]):
            remaining[child] -= 1
            if remaining[child] == 0:
                queue.append(child)
    return colors


def edge_label(row, show_detail_labels=True):
    score = safe_float(row.get('Score', 0.0))
    label = f'Content {score:.4f}'
    if is_candidate_edge(row):
        label = 'Candidate (' + ('filename' if row.get('Evidence_Type') == 'filename_candidate' else 'text reuse') + ')\n' + label
    if show_detail_labels:
        label += '\n' + diff_summary_from_row(row)
        a = safe_float(row.get('Old_Shingle_Containment'), None)
        b = safe_float(row.get('New_Shingle_Containment'), None)
        if a is not None and b is not None and not is_copy_edge(row):
            label += f'\nShared text: {a:.0%} of A / {b:.0%} of B'
    return label

def build_elements(graph, selected_id, direction, depth, layout_mode, show_detail_labels, color_branches=False, time_spacing='compact', include_filename_relatives=False, extra_nodes=None, show_dates=None):
    keep = selected_nodes(graph, selected_id, direction, depth)
    if not keep:
        return []
    keep |= set(extra_nodes or ())
    lineage_keep = set(keep)
    if include_filename_relatives:
        keep |= filename_relatives(graph, selected_id)
    positions = time_positions(graph, keep, selected_id, time_spacing) if layout_mode == 'time' else {}
    colors = branch_colors(graph, keep) if color_branches else {}
    elements = []
    for node_id in sorted(keep):
        row = graph['nodes'].get(node_id, {})
        filename = clean_text(row.get('filename', node_id))
        modified = clean_text(row.get('modified', ''))
        copy_count = int(row.get('copy_count', 1))
        label = node_label(row, node_id, layout_mode, show_dates)
        classes = []
        if node_id == selected_id:
            classes.append('selected-document')
        if copy_count > 1:
            classes.append('duplicate-group')
        item = {'data': {'id': node_id, 'label': label, 'filename': filename, 'modified': modified, 'modified_latest': clean_text(row.get('modified_latest', '')), 'words': clean_text(row.get('words', '')), 'revision': clean_text(row.get('revision', '')), 'creator': clean_text(row.get('creator', '')), 'copy_count': copy_count, 'member_ids': ' | '.join(row.get('member_ids', [])), 'path': row.get('path', ''), 'paths': '\n'.join(row.get('paths', []))}, 'classes': ' '.join(classes)}
        if node_id not in lineage_keep:
            item['classes'] += ' filename-relative'
        item['data']['filename_hint'] = node_id not in lineage_keep
        item['data']['filenames'] = '\n'.join(row.get('filenames', []))
        item['data']['branch_color'] = colors.get(node_id, NEUTRAL_NODE_COLOR)
        if node_id in positions:
            item['position'] = positions[node_id]
        elements.append(item)
    for _, row in graph['edges'].iterrows():
        source = str(row['Parent_ID'])
        target = str(row['Child_ID'])
        if source not in keep or target not in keep:
            continue
        _, gap_hours, chronology = temporal_score(graph, source, target)
        if clean_text(row.get('Chronology_Evidence', '')) == 'conflict':
            chronology = 'conflict'
        elif clean_text(row.get('Chronology_Evidence', '')) == 'equal timestamps':
            chronology = 'ambiguous/autosave'
        if chronology == 'ambiguous/autosave':
            edge_class = 'chronology-ambiguous'
        elif chronology == 'conflict':
            edge_class = 'chronology-conflict'
        else:
            edge_class = 'chronology-normal'
        if is_candidate_edge(row):
            edge_class += ' candidate-edge'
        elements.append({'data': {'evidence_type': clean_text(row.get('Evidence_Type', 'content')), 'candidate_score': clean_text(row.get('Candidate_Score', '')), 'candidate_reason': clean_text(row.get('Candidate_Reason', '')), 'version_progression': clean_text(row.get('Version_Progression', '')), 'filename_date_order': clean_text(row.get('Filename_Date_Order', '')), 'old_shingle_containment': clean_text(row.get('Old_Shingle_Containment', '')), 'new_shingle_containment': clean_text(row.get('New_Shingle_Containment', '')), 'source': source, 'target': target, 'evidence_parent_id': clean_text(row.get('Evidence_Parent_ID', '')), 'evidence_child_id': clean_text(row.get('Evidence_Child_ID', '')), 'score': safe_float(row.get('Score', 0.0)), 'relationship': clean_text(row.get('Relationship', '')), 'label': edge_label(row, show_detail_labels), 'gap': format_gap(gap_hours), 'chronology': chronology, 'evidence_count': safe_int(row.get('Evidence_Edge_Count', 1), 1), 'detail_diff_run': as_bool(row.get('Detail_Diff_Run', False)), 'exact_visible_equal': clean_text(row.get('Exact_Visible_Text_Equal', '')), 'diff_class': clean_text(row.get('Diff_Class', '')), 'paragraph_edit_blocks': clean_text(row.get('Paragraph_Edit_Blocks', '')), 'changed_paragraphs': clean_text(row.get('Changed_Paragraphs_Est', '')), 'diff_edit_blocks': clean_text(row.get('Diff_Edit_Blocks', '')), 'inserted_tokens': clean_text(row.get('Inserted_Tokens', '')), 'deleted_tokens': clean_text(row.get('Deleted_Tokens', '')), 'replaced_old_tokens': clean_text(row.get('Replaced_Old_Tokens', '')), 'replaced_new_tokens': clean_text(row.get('Replaced_New_Tokens', '')), 'changed_tokens': clean_text(row.get('Changed_Tokens_Est', '')), 'token_similarity': clean_text(row.get('Token_Similarity', '')), 'text_delta_pct': clean_text(row.get('Text_Delta_Pct', '')), 'old_content_in_new': clean_text(row.get('Old_Content_In_New', '')), 'new_content_in_old': clean_text(row.get('New_Content_In_Old', '')), 'paragraph_jaccard': clean_text(row.get('Paragraph_Jaccard', '')), 'shingle_jaccard': clean_text(row.get('Shingle_Jaccard', '')), 'changed_old_sample': clean_text(row.get('Changed_Old_Sample', '')), 'changed_new_sample': clean_text(row.get('Changed_New_Sample', ''))}, 'classes': edge_class})
    return elements

def predecessor_table(base_graph, selected_id, min_score=0.0):
    if not selected_id:
        return pd.DataFrame()
    incoming = base_graph['edges'][(base_graph['edges']['Child_ID'] == selected_id) & (base_graph['edges']['Score'] >= min_score)].copy()
    rows = []
    for _, edge_row in incoming.iterrows():
        parent_id = str(edge_row['Parent_ID'])
        parent_row = base_graph['nodes'].get(parent_id, {})
        rank = predecessor_rank(base_graph, parent_id, selected_id, edge_row)
        rows.append({'Evidence_Type': clean_text(edge_row.get('Evidence_Type', 'content')), 'Candidate_Score': clean_text(edge_row.get('Candidate_Score', '')), 'Candidate_Reason': clean_text(edge_row.get('Candidate_Reason', '')), 'Rank_Score': rank['combined'], 'Parent_ID': parent_id, 'Parent_File': clean_text(parent_row.get('filename', parent_id)), 'Copies': int(parent_row.get('copy_count', 1)), 'Parent_Modified': clean_text(parent_row.get('modified', '')), 'Parent_Modified_Latest': clean_text(parent_row.get('modified_latest', '')), 'Time_To_Child': format_gap(rank['gap_hours']), 'Chronology': rank['chronology'], 'Relationship': clean_text(edge_row.get('Relationship', '')), 'Content_Score': safe_float(edge_row.get('Score', 0.0)), 'Word_Similarity': rank['word_similarity'], 'Word_Delta_Pct': rank['word_delta_pct'], 'Old_Content_In_New': safe_float(edge_row.get('Old_Content_In_New', 0.0)), 'New_Content_In_Old': safe_float(edge_row.get('New_Content_In_Old', 0.0)), 'Paragraph_Jaccard': safe_float(edge_row.get('Paragraph_Jaccard', 0.0)), 'Shingle_Jaccard': safe_float(edge_row.get('Shingle_Jaccard', 0.0)), 'Detail_Diff_Run': as_bool(edge_row.get('Detail_Diff_Run', False)), 'Exact_Visible_Text_Equal': clean_text(edge_row.get('Exact_Visible_Text_Equal', '')), 'Diff_Class': clean_text(edge_row.get('Diff_Class', '')), 'Text_Delta_Pct': edge_row.get('Text_Delta_Pct', ''), 'Diff_Edit_Blocks': edge_row.get('Diff_Edit_Blocks', ''), 'Changed_Paragraphs_Est': edge_row.get('Changed_Paragraphs_Est', ''), 'Changed_Tokens_Est': edge_row.get('Changed_Tokens_Est', ''), 'Evidence_Edges': safe_int(edge_row.get('Evidence_Edge_Count', 1), 1), 'Paths': '\n'.join(parent_row.get('paths', []))})
    if not rows:
        return pd.DataFrame()
    result = pd.DataFrame(rows).sort_values(['Rank_Score', 'Content_Score'], ascending=[False, False]).reset_index(drop=True)
    result.index = result.index + 1
    return result

def predecessor_report_text(base_graph, selected_id, min_score=0.0):
    table = predecessor_table(base_graph, selected_id, min_score)
    selected = base_graph['nodes'].get(selected_id, {})
    lines = ['DOCX LINEAGE - LIKELY IMMEDIATE PREDECESSORS', '=' * 78, '', f"Selected logical document: {selected.get('filename', selected_id)}", f'Logical ID:                {selected_id}', f"Physical copies:           {selected.get('copy_count', 1)}", f"Earliest modified:         {selected.get('modified', '')}", f"Latest modified:           {selected.get('modified_latest', '')}", f"Words:                     {selected.get('words', '')}", f"Revision(s):               {selected.get('revision', '')}", '', 'Physical locations:']
    for path in selected.get('paths', []):
        lines.append(f'  - {path}')
    lines.extend(['', f'Chronology tolerance: +/- {TIME_TOLERANCE_HOURS:g} hours', ''])
    if table.empty:
        lines.append('No incoming candidate parent edges were found.')
        return '\n'.join(lines)
    lines.extend(['LIKELY PREVIOUS VERSIONS', '-' * 78])
    for rank_no, row in table.iterrows():
        delta = row['Word_Delta_Pct']
        delta_text = f'{delta:+.2f}%' if pd.notna(delta) else 'unknown'
        text_delta = row['Text_Delta_Pct']
        text_delta_text = f'{float(text_delta):.4f}%' if text_delta != '' and (not pd.isna(text_delta)) else 'not run'
        edits = row['Diff_Edit_Blocks']
        edits_text = str(safe_int(edits)) if edits != '' and (not pd.isna(edits)) else 'not run'
        changed_paras = row['Changed_Paragraphs_Est']
        changed_paras_text = str(safe_int(changed_paras)) if changed_paras != '' and (not pd.isna(changed_paras)) else 'not run'
        lines.extend([f"{rank_no}. {row['Parent_File']}", f"   Physical copies:        {row['Copies']}", f"   Rank score:             {row['Rank_Score']:.3f}", f"   Content score:          {row['Content_Score']:.3f}", f"   Evidence type:          {row['Evidence_Type']}", f"   Candidate rank:         {row['Candidate_Score']} (heuristic, not probability)", f"   Candidate evidence:     {row['Candidate_Reason']}", f"   Relationship:           {row['Relationship']}", f"   Earliest modified:      {row['Parent_Modified']}", f"   Latest modified:        {row['Parent_Modified_Latest']}", f"   Time to selected:       {row['Time_To_Child']}", f"   Chronology:             {row['Chronology']}", f"   Word similarity:        {row['Word_Similarity']:.3f}", f'   Word-count change:      {delta_text}', f"   Detailed diff class:    {row['Diff_Class'] or 'not run'}", f'   Text delta:             {text_delta_text}', f'   Diff edit blocks:       {edits_text}', f'   Changed paragraphs est: {changed_paras_text}', f"   Old content in new:     {row['Old_Content_In_New']:.3f}", f"   New content in old:     {row['New_Content_In_Old']:.3f}", f"   Paragraph Jaccard:      {row['Paragraph_Jaccard']:.3f}", f"   Shingle Jaccard:        {row['Shingle_Jaccard']:.3f}", '   Locations:'])
        for path in str(row['Paths']).splitlines():
            lines.append(f'      - {path}')
        lines.append('')
    lines.extend(['INTERPRETATION', '-' * 78, 'The lineage rank is evidence-based, not proof of exact save order.', 'For very similar edges, Text_Delta_Pct and Diff_Edit_Blocks expose', 'small edits that the broad lineage score may round to 1.00.', "Diff_Edit_Blocks is not Microsoft Word's revision count; it is the", 'SequenceMatcher edit-block count generated by the lineage builder.'])
    return '\n'.join(lines)
def latest_family(row):
    family = clean_text(row.get('Filename_Family', '')).strip()
    if family:
        return family
    stem = os.path.splitext(clean_text(row.get('Filename', '')))[0].casefold()
    stem = re.sub(r'(?:[_ .-]+v\d+)+$', '', stem)
    def date_token(match):
        for fmt in ('%Y%m%d', '%d%m%Y'):
            value = pd.to_datetime(match.group(), format=fmt, errors='coerce')
            if not pd.isna(value) and 1990 <= value.year <= 2100:
                return ' '
        return match.group()
    stem = re.sub(r'(?<!\d)\d{8}(?!\d)', date_token, stem)
    return ' '.join(re.findall(r'[^\W_]+', stem)) or clean_text(row.get('ID', ''))


@lru_cache(maxsize=1)
def latest_version_index():
    states, physical_to_state = {}, {}
    for _, row in nodes.iterrows():
        text_hash = clean_hash(row.get('Text_SHA256', ''))
        # Empty/unreadable text must never join unrelated documents.
        usable = bool(text_hash and safe_float(row.get('Words', 0)) > 0)
        key = 'text:' + text_hash if usable else 'file:' + str(row['ID'])
        physical_to_state[str(row['ID'])] = key
        states.setdefault(key, {'members': [], 'families': set(), 'usable': usable})
        states[key]['members'].append(row.to_dict())
        states[key]['families'].add(latest_family(row))
    neighbors = {key: set() for key in states}
    family_states = defaultdict(set)
    for key, state in states.items():
        members = sorted(state['members'], key=lambda r: (pd.isna(r['_Modified_DT']), r['_Modified_DT'] if not pd.isna(r['_Modified_DT']) else pd.Timestamp.max, str(r['ID'])))
        state['members'] = members
        valid = [r for r in members if not pd.isna(r['_Modified_DT'])]
        state['first'] = valid[0]['_Modified_DT'] if valid else pd.NaT
        state['latest'] = valid[-1] if valid else members[0]
        state['later'] = set()
        state['possible'] = set()
        state['ambiguous'] = set()
        state['incoming'] = set()
        for family in state['families']:
            family_states[family].add(key)
    for _, edge in edges.iterrows():
        source = physical_to_state.get(str(edge['Parent_ID']))
        target = physical_to_state.get(str(edge['Child_ID']))
        if source not in states or target not in states or source == target:
            continue
        neighbors[source].add(target)
        neighbors[target].add(source)
        states[target]['incoming'].add(source)
        a, b = states[source], states[target]
        chronology = clean_text(edge.get('Chronology_Evidence', ''))
        if (pd.isna(a['first']) or pd.isna(b['first']) or b['first'] <= a['first']
                or chronology in {'conflict', 'equal timestamps'}):
            a['ambiguous'].add(target)
            continue
        relationship = clean_text(edge.get('Relationship', '')).lower()
        # Excerpts/partial reuse/expansion into a larger file do not establish
        # that the source document has been superseded as its own document.
        if (not is_candidate_edge(edge) and not as_bool(edge.get('Uncertain_Lineage', False))
                and relationship == 'revision' and a['usable'] and b['usable']):
            a['later'].add(target)
        else:
            a['possible'].add(target)
    return states, physical_to_state, neighbors, family_states


@lru_cache(maxsize=128)
def latest_version_records(selected_original_id=None, scope='related', show_previous=False):
    """Read all exported evidence independently of graph layout/filter settings.

    Collapse equal normalized text for this table only. A late copy cannot
    advance the date of a text state or establish a new text revision.
    """
    if scope == 'related' and not selected_original_id:
        return []
    states, physical_to_state, neighbors, family_states = latest_version_index()
    keep = set(states)
    selected_key = physical_to_state.get(str(selected_original_id))
    if scope == 'related':
        if selected_key is None:
            return []
        keep = {selected_key}
        queue = deque([selected_key])
        while queue:
            key = queue.popleft()
            linked = set(neighbors[key])
            for family in states[key]['families']:
                linked.update(family_states[family])
            for neighbor in linked - keep:
                keep.add(neighbor)
                queue.append(neighbor)
    records = []
    for key in keep:
        state = states[key]
        first_row, latest_row = state['members'][0], state['latest']
        if state['later']:
            status = 'Later text revision linked'
        elif not state['usable']:
            status = 'Text unavailable — review'
        elif state['ambiguous']:
            status = 'Chronology unclear — review'
        elif state['possible']:
            status = 'Possible later revision / reuse — review'
        elif not state['incoming']:
            status = 'No linked revisions — review'
        else:
            status = 'Likely latest text version'
        if state['later'] and not show_previous:
            continue
        copy_note = 'One physical file'
        if len(state['members']) > 1:
            copy_note = 'Same normalized text in multiple files; not a new text revision'
        if state['usable'] and not pd.isna(state['first']) and latest_row['_Modified_DT'] > state['first']:
            copy_note = 'Newer file timestamp, same normalized text'
        def timestamp(value):
            return '' if pd.isna(value) else pd.Timestamp(value).strftime('%Y-%m-%d %H:%M:%S')
        def target_names(keys):
            return '; '.join(sorted({clean_text(states[k]['members'][0].get('Filename', '')) for k in keys}))
        records.append({
            'Family': latest_family(first_row), 'Status': status,
            'Text version': clean_text(first_row.get('Filename', '')),
            'Text first file time': timestamp(state['first']),
            'Newest file': clean_text(latest_row.get('Filename', '')),
            'Newest file time': timestamp(latest_row['_Modified_DT']),
            'Words': clean_text(first_row.get('Words', '')), 'Files': len(state['members']),
            'Copy / revision': copy_note,
            'Later revisions': target_names(state['later']),
            'Possible later / reused in': target_names(state['possible']),
            'Chronology to review': target_names(state['ambiguous']),
            'Newest location': clean_text(latest_row.get('Full_Path', '')),
            'Evidence': f"{len(state['incoming'])} incoming text-state link(s); {len(state['later'])} later revision(s); {len(state['possible'])} possible revision/reuse link(s).",
            '_members': state['members'], '_state': key,
        })
    # Sort within family by earliest known time of the text, never newest-copy time.
    records.sort(key=lambda r: (r['Text first file time'], r['Text version']), reverse=True)
    records.sort(key=lambda r: r['Family'])
    return records


def render_latest_versions(selected_original_id, scope, options):
    records = latest_version_records(selected_original_id, scope, 'previous' in (options or []))
    if scope == 'related' and not selected_original_id:
        return 'Choose a document above, or switch to All document families.'
    if not records:
        return 'No versions found in the loaded CSVs for this selection.'
    columns = ['Family', 'Status', 'Text version', 'Text first file time', 'Newest file', 'Newest file time', 'Words', 'Files', 'Copy / revision', 'Evidence', 'Later revisions', 'Possible later / reused in', 'Chronology to review', 'Newest location']
    header = html.Thead(html.Tr([html.Th(c, style={'padding': '6px', 'textAlign': 'left'}) for c in columns] + [html.Th('All files / locations')]))
    body = []
    for record in records:
        members = [html.Div([html.B(clean_text(r.get('Filename', '')) + ' | ' + clean_text(r.get('Modified', ''))), html.Div(clean_text(r.get('Full_Path', ''))), html.Small('Physical ID: ' + str(r['ID']))], style={'marginBottom': '8px'}) for r in record['_members']]
        cells = [html.Td(str(record[c]), style={'padding': '6px', 'verticalAlign': 'top', 'borderBottom': '1px solid #ddd'}) for c in columns]
        cells.append(html.Td(html.Details([html.Summary('Show files'), *members])))
        body.append(html.Tr(cells))
    return html.Div([html.P(f'{len(records)} text-state candidate(s). Multiple branches remain separate; no file is automatically declared final.'), html.Div(html.Table([header, html.Tbody(body)], style={'borderCollapse': 'collapse', 'fontSize': '12px'}), style={'overflowX': 'auto'})])


# Comparison reads current files on demand; it never changes the lineage CSVs.
COMPARE_MAX_XML_BYTES = 20 * 1024 * 1024
COMPARE_MAX_TOKENS = 12000
COMPARE_MAX_BLOCKS = 80
COMPARE_EXCERPT_CHARS = 2400


def extract_body_paragraph_records(xml_bytes):
    """Current body/table/text-box text; locations are 1-based XML paragraph ordinals.

    Includes insertions and moved-to text; excludes deletions and moved-from text.
    Empty/deleted paragraphs still occupy XML ordinals. These are not page numbers.
    Nested text-box paragraphs are emitted once, separately from their container.
    """
    w = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'
    root = ET.fromstring(xml_bytes)
    ordinals = {id(p): i for i, p in enumerate(root.iter(w+'p'), 1)}
    records = []

    def pieces(element):
        if element.tag in {w+'del', w+'moveFrom'}:
            return []
        if element.tag == w+'t':
            return [element.text or '']
        if element.tag in {w+'tab', w+'br', w+'cr'}:
            return [' ']
        result = []
        for child in element:
            if child.tag != w+'p':
                result.extend(pieces(child))
        return result

    def walk(element):
        if element.tag in {w+'del', w+'moveFrom'}:
            return
        if element.tag == w+'p':
            text = re.sub(r'\s+', ' ', ''.join(pieces(element)).replace('\xa0', ' ')).strip()
            if text:
                records.append({'xml_paragraph_index': ordinals[id(element)], 'text': text})
        for child in element:
            walk(child)
    walk(root)
    return records


def read_comparison_docx(path):
    w = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'
    with open(path, 'rb') as stream:
        before = os.fstat(stream.fileno())
        with zipfile.ZipFile(stream) as archive:
            member = archive.getinfo('word/document.xml')
            if member.file_size > COMPARE_MAX_XML_BYTES:
                raise ValueError('Document text XML exceeds the comparison size limit.')
            xml_bytes = archive.read(member)
            root = ET.fromstring(xml_bytes)
        after = os.fstat(stream.fileno())
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise ValueError('The document changed while it was being read. Please compare again.')
    paragraphs = [record['text'] for record in extract_body_paragraph_records(xml_bytes)]
    tracked = any(e.tag in {w+'ins', w+'del', w+'moveFrom', w+'moveTo'} for e in root.iter())
    if not paragraphs:
        raise ValueError('No comparable body text was extracted; this is not evidence of identical documents.')
    text = '\n'.join(paragraphs)
    tokens = re.findall(r'\w+|[^\w\s]', text, flags=re.UNICODE)
    if len(tokens) > COMPARE_MAX_TOKENS:
        raise ValueError(f'Document exceeds the {COMPARE_MAX_TOKENS:,}-token interactive comparison limit. Increase COMPARE_MAX_TOKENS only if longer comparisons are acceptable.')
    return {'paragraphs': paragraphs, 'text': text, 'tokens': tokens,
            'words': len(re.findall(r'\w+', text, flags=re.UNICODE)),
            'modified': pd.Timestamp(before.st_mtime, unit='s').strftime('%Y-%m-%d %H:%M:%S UTC'),
            'tracked': tracked}


def compare_body_texts(old, new):
    matcher = difflib.SequenceMatcher(None, old['tokens'], new['tokens'], autojunk=False)
    added = removed = edits = 0
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != 'equal':
            edits += 1
            removed += i2-i1
            added += j2-j1
    blocks = []
    retained_blocks = []
    retained_paragraphs = 0
    inserted = deleted = replaced_old = replaced_new = total_blocks = 0
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, old['paragraphs'], new['paragraphs'], autojunk=False).get_opcodes():
        if tag == 'equal':
            retained_paragraphs += i2-i1
            if len(retained_blocks) < COMPARE_MAX_BLOCKS:
                text = '\n'.join(old['paragraphs'][i1:i2])
                retained_blocks.append({'tag': 'retained', 'old_range': (i1+1, i2),
                    'new_range': (j1+1, j2), 'old': text[:COMPARE_EXCERPT_CHARS],
                    'new': text[:COMPARE_EXCERPT_CHARS], 'truncated': len(text) > COMPARE_EXCERPT_CHARS})
            continue
        total_blocks += 1
        if tag == 'insert': inserted += j2-j1
        elif tag == 'delete': deleted += i2-i1
        else:
            replaced_old += i2-i1
            replaced_new += j2-j1
        if len(blocks) < COMPARE_MAX_BLOCKS:
            a = '\n'.join(old['paragraphs'][i1:i2])
            b = '\n'.join(new['paragraphs'][j1:j2])
            blocks.append({'tag': tag, 'old_range': (i1+1,i2), 'new_range': (j1+1,j2),
                           'old': a[:COMPARE_EXCERPT_CHARS], 'new': b[:COMPARE_EXCERPT_CHARS],
                           'truncated': max(len(a),len(b)) > COMPARE_EXCERPT_CHARS})
    return {'equal': old['paragraphs']==new['paragraphs'], 'similarity': matcher.ratio(),
            'added': added, 'removed': removed, 'edit_blocks': edits,
            'paragraphs_added': inserted, 'paragraphs_removed': deleted,
            'paragraphs_replaced_old': replaced_old, 'paragraphs_replaced_new': replaced_new,
            'changed_paragraphs_est': inserted+deleted+max(replaced_old,replaced_new),
            'blocks': blocks, 'total_blocks': total_blocks,
            'retained_blocks': retained_blocks, 'retained_paragraphs': retained_paragraphs}


def comparison_highlights(old_text, new_text):
    # Dash renders strings as text, never as raw HTML from the document.
    a = re.findall(r'\w+|[^\w\s]|\s+', old_text)
    b = re.findall(r'\w+|[^\w\s]|\s+', new_text)
    left, right = [], []
    for tag,i1,i2,j1,j2 in difflib.SequenceMatcher(None,a,b,autojunk=False).get_opcodes():
        if i2>i1:
            left.append(html.Span(''.join(a[i1:i2]), style={} if tag=='equal' else {'backgroundColor':'#ffe1e1','textDecoration':'line-through'}))
        if j2>j1:
            right.append(html.Span(''.join(b[j1:j2]), style={} if tag=='equal' else {'backgroundColor':'#dff3df'}))
    return left or ['(No text in this block)'], right or ['(No text in this block)']


def passage_columns(block):
    left, right = comparison_highlights(block['old'], block['new'])
    columns = []
    for label, parts, bounds in (('A — before', left, block['old_range']), ('B — after', right, block['new_range'])):
        location = f'paragraphs {bounds[0]}–{bounds[1]}' if bounds[1] >= bounds[0] else 'no paragraphs'
        columns.append(html.Div([html.B(label + ' | ' + location), html.Pre(parts,
            style={'whiteSpace': 'pre-wrap', 'overflowWrap': 'anywhere', 'fontFamily': 'inherit'})],
            style={'flex': '1', 'minWidth': '260px', 'padding': '10px', 'border': '1px solid #ddd'}))
    return html.Div(columns, style={'display': 'flex', 'flexWrap': 'wrap', 'gap': '10px'})


def compare_selected_files(n_clicks, old_id, new_id):
    if not n_clicks:
        return 'Select two physical files and click Compare.'
    if not old_id or not new_id:
        return 'Please select both a before file and an after file.'
    if old_id == new_id:
        return 'Select two different physical files.'
    chosen = []
    for node_id in (old_id, new_id):
        matches = nodes[nodes['ID']==str(node_id)]
        if len(matches)!=1:
            return 'A selected file is no longer uniquely available in the loaded CSV. Restart Dash with the current CSVs.'
        chosen.append(matches.iloc[0])
    documents = []
    for row in chosen:
        path = clean_text(row.get('Full_Path',''))
        if not path.lower().endswith('.docx'):
            return 'The selected row does not contain a DOCX path: ' + path
        try:
            documents.append(read_comparison_docx(path))
        except (OSError, zipfile.BadZipFile, KeyError, ET.ParseError, ValueError, RuntimeError) as exc:
            return html.Div([html.B('Could not compare this file: '), html.Div(path), html.Div(str(exc))])
    old,new = documents
    result = compare_body_texts(old,new)
    content = [html.H4('Comparison results: A (before) → B (after)'), html.P('Current files read from disk. Graph scores describe the loaded CSV snapshot and may differ after later edits. Only extracted body and table text is compared; formatting, images, headers, footers, footnotes and comments are excluded.')]
    for label,row,doc in zip(('A — before','B — after'),chosen,documents):
        content.append(html.Div([html.B(label + ': ' + clean_text(row.get('Filename',''))), html.Div(clean_text(row.get('Full_Path',''))), html.Small('Read from disk; modified ' + doc['modified'] + f"; {doc['words']:,} words; {len(doc['paragraphs']):,} nonempty paragraphs.")],style={'marginBottom':'10px'}))
    content.append(html.P('Identical extracted body text (case and punctuation preserved; whitespace normalized).' if result['equal'] else 'Extracted body text differs. Before/after follows your selection, not inferred chronology.'))
    if old['tracked'] or new['tracked']:
        content.append(html.P('Tracked changes detected: inserted text is included; deleted and moved-from text is omitted for this comparison.'))
    content.append(html.P(f"Token similarity: {result['similarity']:.1%}. Added tokens: {result['added']:,}; removed tokens: {result['removed']:,}; edit blocks: {result['edit_blocks']:,}. Tokens include punctuation; replacements count on both sides."))
    content.append(html.P(f"Changed paragraphs (estimate): {result['changed_paragraphs_est']:,}. Added: {result['paragraphs_added']}; removed: {result['paragraphs_removed']}; replaced: {result['paragraphs_replaced_old']} before / {result['paragraphs_replaced_new']} after. These are comparison counts, not Word revision counts."))
    for index,block in enumerate(result['blocks'],1):
        content.append(html.Details([html.Summary(f"Change {index}: {block['tag']}" + (' — excerpt truncated' if block['truncated'] else '')), passage_columns(block)], open=index<=3))
    if result['retained_blocks']:
        passages = [html.P('Matching passages in sequence; locations count nonempty extracted paragraphs. Excerpts and the number of displayed blocks are limited.')]
        for block in result['retained_blocks']:
            if block['truncated']:
                passages.append(html.Small('Retained excerpt truncated'))
            passages.append(passage_columns(block))
        content.append(html.Details([html.Summary(f"Retained passages: {result['retained_paragraphs']} unchanged paragraphs"), *passages]))
    if result['total_blocks']>len(result['blocks']):
        content.append(html.P(f"Showing the first {len(result['blocks'])} of {result['total_blocks']} changed blocks; counts above cover the full extracted text."))
    return html.Div(content)


def compare_edge_files(data, elements):
    if not data:
        return 'Click an edge to compare its source (A) and target (B).'
    pair = (clean_text(data.get('evidence_parent_id')), clean_text(data.get('evidence_child_id')))
    if not all(pair):
        return 'This edge has no physical file pair. Rerun the Dash cell to load edge comparisons.'
    matching = any(
        e.get('data', {}).get('source') == data.get('source')
        and e.get('data', {}).get('target') == data.get('target')
        and (e.get('data', {}).get('evidence_parent_id'), e.get('data', {}).get('evidence_child_id')) == pair
        for e in (elements or []))
    if not matching:
        return 'The graph changed. Click an edge in the current view.'
    return html.Div([html.P(f'Comparing the evidence files {pair[0]} → {pair[1]}. A follows the edge source and B the target; inferred chronology may be uncertain.'),
                     compare_selected_files(1, *pair)])


document_options = []
for _, row in nodes.sort_values('_Modified_DT', ascending=False, na_position='last').iterrows():
    document_options.append({'label': f"{row.get('Modified', '')}  |  {row.get('Filename', '')}  |  {row['ID']}  |  {row.get('Full_Path', '')}", 'value': str(row['ID'])})
# Target-specific passage search. Independent of lineage edges and graph filters.
PASSAGE_CACHE_FILE = os.path.join(OUTPUT_DIR, f'{RUN_NAME}_text_cache.json.gz')
PASSAGE_SEED_WORDS = 5
PASSAGE_MIN_WORDS = 24
PASSAGE_MIN_SIMILARITY = 0.85
PASSAGE_MAX_EDIT_GAP = 12
PASSAGE_MAX_SEED_OCCURRENCES = 100
PASSAGE_MAX_ANCHORS_PER_FILE = 200000
PASSAGE_MAX_VERIFY_WORDS = 1500
PASSAGE_COMMON_FRACTION = 0.25
PASSAGE_DISPLAY_BLOCKS = 20
PASSAGE_PREVIEW_WORDS = 400


def passage_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


# Bind the cache to the CSV snapshot actually loaded by this dashboard.
PASSAGE_NODES_SHA256 = passage_sha256(NODES_FILE)


@lru_cache(maxsize=2)
def read_passage_snapshot(path, size, mtime_ns, nodes_sha256):
    with gzip.open(path, 'rt', encoding='utf-8') as stream:
        snapshot = json.load(stream)
    current = os.stat(path)
    if (current.st_size, current.st_mtime_ns) != (size, mtime_ns):
        raise ValueError('Text cache changed while loading. Retry the search.')
    versions = (snapshot.get('schema_version'), snapshot.get('extraction_version'),
                snapshot.get('normalization_version'))
    if versions != (1, 'body-visible-v1', 'lowercase-unicode-words-v1'):
        raise ValueError('Incompatible text cache. Rerun the collection cell.')
    if snapshot.get('nodes_sha256') != nodes_sha256:
        raise ValueError('Text cache and loaded nodes CSV are from different scans. Rerun collection and restart the dashboard.')
    docs = snapshot.get('documents')
    if not isinstance(docs, list):
        raise ValueError('Invalid text cache: documents are missing.')
    seen = set()
    for doc in docs:
        if doc['id'] in seen:
            raise ValueError('Invalid text cache: duplicate document IDs.')
        seen.add(doc['id'])
        offset = 0
        ordinal = 0
        for para in doc['paragraphs']:
            words = re.sub(r'[^\w]+', ' ', para['text'].lower().replace('\xa0', ' ')).split()
            if (words != para['normalized_words'] or para['word_start'] != offset
                    or para['word_end'] != offset + len(words)
                    or para['xml_paragraph_index'] <= ordinal):
                raise ValueError('Invalid text cache: paragraph locations or normalized words disagree. Rerun collection.')
            offset = para['word_end']
            ordinal = para['xml_paragraph_index']
        if offset != doc['word_count']:
            raise ValueError('Invalid text cache: word counts disagree.')
    return snapshot


def passage_words(doc):
    return [word for para in doc['paragraphs'] for word in para['normalized_words']]


def passage_intervals(spans):
    """Union half-open spans so overlaps and repeated alternatives count once."""
    result = []
    for start, end in sorted(spans):
        if end <= start:
            continue
        if result and start <= result[-1][1]:
            result[-1][1] = max(end, result[-1][1])
        else:
            result.append([start, end])
    return result


def passage_length(spans):
    return sum(end - start for start, end in passage_intervals(spans))


def passage_location(doc, start, end):
    paragraphs = [p for p in doc['paragraphs'] if p['word_start'] < end and p['word_end'] > start]
    return [p['xml_paragraph_index'] for p in paragraphs]


def passage_seed_index(words):
    index = defaultdict(list)
    for pos in range(len(words) - PASSAGE_SEED_WORDS + 1):
        index[tuple(words[pos:pos + PASSAGE_SEED_WORDS])].append(pos)
    return index


def passage_pair_matches(target_words, source_words, target_index, min_words, similarity):
    """Find local exact runs, then bridge small edits without global ordering assumptions.

    Five-word anchors are a lexical heuristic: a rewrite without a shared anchor
    is not detected. Reordered sections are independent runs. Verification uses
    equal-word spans, not the enclosing window, for target coverage.
    """
    source_index = passage_seed_index(source_words)
    diagonals = defaultdict(list)
    anchors = 0
    skipped_seeds = 0
    for seed in target_index.keys() & source_index.keys():
        targets, sources = target_index[seed], source_index[seed]
        if max(len(targets), len(sources)) > PASSAGE_MAX_SEED_OCCURRENCES:
            skipped_seeds += 1
            continue
        anchors += len(targets) * len(sources)
        if anchors > PASSAGE_MAX_ANCHORS_PER_FILE:
            return [], {'limited': True, 'skipped_seeds': skipped_seeds,
                        'reason': 'Too many repeated anchors; file not fully evaluated.'}
        for t in targets:
            for s in sources:
                diagonals[s-t].append(t)
    runs = []
    for delta, positions in diagonals.items():
        spans = passage_intervals((t, t + PASSAGE_SEED_WORDS) for t in positions)
        runs.extend((t, end, t+delta, end+delta) for t, end in spans)
    runs.sort()
    # Each exact run is useful independently, even if another section moved.
    candidates = set(run for run in runs if run[1]-run[0] >= min_words)
    # Connect nearby runs in both coordinates. Bound local verification sizes.
    starts = defaultdict(list)
    for run in runs:
        starts[run[0]].append(run)
    consumed = set()
    for first in runs:
        if first in consumed:
            continue
        t0, t1, s0, s1 = first
        chain = [first]
        chain_candidates = []
        while True:
            neighbors = []
            for next_t in range(t1, t1 + PASSAGE_MAX_EDIT_GAP + 1):
                for run in starts.get(next_t, ()):
                    if s1 <= run[2] <= s1 + PASSAGE_MAX_EDIT_GAP and run[1] > t1:
                        if max(run[1]-t0, run[3]-s0) <= PASSAGE_MAX_VERIFY_WORDS:
                            neighbors.append(run)
            if not neighbors:
                break
            # Prefer aligned nearby anchors over distant repeated occurrences.
            nxt = min(neighbors, key=lambda r: (abs((r[0]-t1)-(r[2]-s1)),
                                                r[0]-t1+r[2]-s1, -(r[1]-r[0]), r))
            t1, s1 = nxt[1], nxt[3]
            chain.append(nxt)
            if min(t1-t0, s1-s0) >= min_words:
                chain_candidates.append((t0, t1, s0, s1))
        # Consume a chain only up to a verified prefix. If a noisy bridge fails,
        # later runs get their own chance instead of being swallowed by that bridge.
        for window in reversed(chain_candidates):
            x0, x1, y0, y1 = window
            aa, bb = target_words[x0:x1], source_words[y0:y1]
            matched = sum(m.size for m in difflib.SequenceMatcher(None, aa, bb, autojunk=False).get_matching_blocks())
            if matched >= min_words and matched/max(len(aa), len(bb)) >= similarity:
                candidates.add(window)
                consumed.update(run for run in chain if run[1] <= x1 and run[3] <= y1)
                break
    accepted = []
    for t0, t1, s0, s1 in sorted(candidates, key=lambda r: (-(r[1]-r[0]), r)):
        if any(t0 >= b['target_start'] and t1 <= b['target_end']
               and s0 >= b['source_start'] and s1 <= b['source_end'] for b in accepted):
            continue
        a, b = target_words[t0:t1], source_words[s0:s1]
        if a == b:
            equal = [(0, 0, len(a))]
        else:
            equal = [(x.a, x.b, x.size) for x in difflib.SequenceMatcher(None, a, b, autojunk=False).get_matching_blocks() if x.size]
        matched = sum(n for _, _, n in equal)
        score = matched / max(len(a), len(b), 1)
        if matched < min_words or score < similarity:
            continue
        accepted.append({'target_start': t0, 'target_end': t1,
                         'source_start': s0, 'source_end': s1,
                         'similarity': score, 'matched_words': matched,
                         'kind': 'exact normalized wording' if score == 1 else 'lightly edited wording',
                         'target_equal_spans': [[t0+x, t0+x+n] for x, _, n in equal],
                         'source_equal_spans': [[s0+y, s0+y+n] for _, y, n in equal]})
    return sorted(accepted, key=lambda b: (b['target_start'], b['source_start'])), {
        'limited': bool(skipped_seeds), 'skipped_seeds': skipped_seeds,
        'reason': 'Highly repeated anchors omitted.' if skipped_seeds else ''}


def search_passage_documents(target, documents, min_words=PASSAGE_MIN_WORDS,
                             similarity=PASSAGE_MIN_SIMILARITY):
    """Pure search over usable snapshot records; all results retained for later tree integration."""
    if min_words < PASSAGE_SEED_WORDS or not 0.5 <= similarity <= 1:
        raise ValueError('Invalid passage thresholds.')
    target_words = passage_words(target)
    target_index = passage_seed_index(target_words)
    prepared = [(d, passage_words(d)) for d in documents if d['id'] != target['id']]
    # Count document frequency once per source, not once per occurrence.
    frequency = defaultdict(int)
    for _, words in prepared:
        for seed in set(tuple(words[i:i+PASSAGE_SEED_WORDS]) for i in range(len(words)-PASSAGE_SEED_WORDS+1)) & target_index.keys():
            frequency[seed] += 1
    weights = [0.0] * len(target_words)
    common_limit = max(3, len(prepared) * PASSAGE_COMMON_FRACTION)
    for seed, positions in target_index.items():
        count = frequency[seed]
        weight = 1.0 if count < common_limit else 1.0 / max(count, 1)
        for start in positions:
            for pos in range(start, start+PASSAGE_SEED_WORDS):
                weights[pos] = max(weights[pos], weight)
    results, limits = [], []
    pair_cache = {}
    for source, words in prepared:
        key = tuple(words)
        if key not in pair_cache:
            if len(pair_cache) >= 64:
                pair_cache.pop(next(iter(pair_cache)))
            pair_cache[key] = passage_pair_matches(target_words, words, target_index, min_words, similarity)
        raw_blocks, limit = pair_cache[key]
        blocks = [dict(block) for block in raw_blocks]  # Locations belong to this physical file.
        if limit['limited']:
            limits.append({'id': source['id'], 'filename': source['filename'], **limit})
        if not blocks:
            continue
        for block in blocks:
            block['target_paragraphs'] = passage_location(target, block['target_start'], block['target_end'])
            block['source_paragraphs'] = passage_location(source, block['source_start'], block['source_end'])
        target_spans = passage_intervals(span for b in blocks for span in b['target_equal_spans'])
        source_spans = passage_intervals(span for b in blocks for span in b['source_equal_spans'])
        covered = passage_length(target_spans)
        weighted = sum(sum(weights[a:b]) for a, b in target_spans)
        results.append({'source_id': source['id'], 'filename': source['filename'], 'path': source['path'],
                        'blocks': blocks, 'target_equal_spans': target_spans,
                        'matched_target_words': covered,
                        'target_coverage': covered/max(len(target_words), 1),
                        'source_coverage': passage_length(source_spans)/max(len(words), 1),
                        'weighted_words': weighted, 'common_wording': weighted < covered * 0.5,
                        'later_timestamp': source['fingerprint']['mtime_ns'] > target['fingerprint']['mtime_ns'],
                        'limited': limit['limited']})
    results.sort(key=lambda r: (-r['weighted_words'], -r['matched_target_words'], r['path'], r['source_id']))
    return {'target_id': target['id'], 'target_filename': target['filename'],
            'target_path': target['path'], 'min_words': min_words, 'similarity': similarity,
            'target_word_weights': weights,
            'searched_sources': len(prepared), 'sources': results, 'limits': limits}


def run_target_passage_search(target_id, min_words, similarity):
    stat = os.stat(PASSAGE_CACHE_FILE)
    snapshot = read_passage_snapshot(PASSAGE_CACHE_FILE, stat.st_size, stat.st_mtime_ns, PASSAGE_NODES_SHA256)
    rows = {str(row['ID']): row for _, row in nodes.iterrows()}
    usable, skipped = [], []
    for doc in snapshot['documents']:
        reason = ''
        row = rows.get(doc['id'])
        if row is None or os.path.normcase(os.path.abspath(str(row['Full_Path']))) != os.path.normcase(doc['path']):
            raise ValueError('Cache file IDs/paths do not match the loaded dataset. Restart after collection.')
        if str(row['Binary_SHA256']) != doc['fingerprint']['binary_sha256']:
            raise ValueError('Cache fingerprints do not match the loaded dataset. Rerun collection.')
        if doc['status'] != 'ok':
            reason = doc['status'] + ': ' + doc.get('error', '')
        else:
            try:
                current = os.stat(doc['path'])
                if (current.st_size, current.st_mtime_ns) != (doc['fingerprint']['size_bytes'], doc['fingerprint']['mtime_ns']):
                    reason = 'Changed since collection; rerun collection.'
            except OSError:
                reason = 'File unavailable.'
        if reason:
            skipped.append({'id': doc['id'], 'filename': doc['filename'], 'reason': reason})
        else:
            usable.append(doc)
    if set(rows) != {d['id'] for d in snapshot['documents']}:
        raise ValueError('Cache document set does not match the loaded dataset.')
    target = next((d for d in usable if d['id'] == str(target_id)), None)
    if target is None:
        raise ValueError('Selected target has no usable current cached text. Rerun collection; check that the file is accessible and contains body text.')
    result = search_passage_documents(target, usable, int(min_words), float(similarity))
    result['skipped'] = skipped
    result['snapshot_created_utc'] = snapshot['created_utc']
    result['cache_signature'] = (stat.st_size, stat.st_mtime_ns)
    docs = {d['id']: d for d in usable}
    result['contributor_groups'] = build_contributor_groups(result, docs, edges.to_dict('records'))
    return result, docs


def passage_context(doc, start, end):
    paras = [p for p in doc['paragraphs'] if p['word_start'] < end and p['word_end'] > start]
    text = '\n'.join(p['text'] for p in paras)
    return text[:2400] + ('\n[Paragraph context truncated]' if len(text) > 2400 else '')


def render_passage_source(source, target, doc, open_details=False, include_context='report'):
    notes = (' | later file timestamp; origin uncertain' if source['later_timestamp'] else '')
    notes += ' | mostly common wording' if source['common_wording'] else ''
    notes += ' | search limited' if source['limited'] else ''
    passages = [include_matching_file_button(source['source_id'], include_context), matching_file_status(source['source_id'], include_context), html.P(source['path']), html.P(
        f"Showing {min(PASSAGE_DISPLAY_BLOCKS, len(source['blocks']))} of {len(source['blocks'])} blocks. "
        'Word offsets below are zero-based, end-exclusive. Highlighted text is normalized wording; original paragraph context is expandable.')]
    words = passage_words(doc)
    target_words = passage_words(target)
    for block in source['blocks'][:PASSAGE_DISPLAY_BLOCKS]:
        source_end = min(block['source_end'], block['source_start'] + PASSAGE_PREVIEW_WORDS)
        target_end = min(block['target_end'], block['target_start'] + PASSAGE_PREVIEW_WORDS)
        left, right = comparison_highlights(' '.join(words[block['source_start']:source_end]),
                                            ' '.join(target_words[block['target_start']:target_end]))
        preview_note = (' · Preview truncated to 400 words per side' if
            source_end < block['source_end'] or target_end < block['target_end'] else '')
        columns = []
        for label, parts, document, start, end, paragraphs in (
            ('Source', left, doc, block['source_start'], block['source_end'], block['source_paragraphs']),
            ('Target', right, target, block['target_start'], block['target_end'], block['target_paragraphs'])):
            columns.append(html.Div([html.B(f'{label}: XML paragraphs {paragraphs}; words [{start}, {end})'),
                html.Pre(parts, style={'whiteSpace': 'pre-wrap', 'overflowWrap': 'anywhere'}),
                html.Details([html.Summary('Original paragraph context'), html.Pre(passage_context(document, start, end),
                    style={'whiteSpace': 'pre-wrap', 'overflowWrap': 'anywhere'})])],
                style={'flex': '1', 'minWidth': '280px', 'padding': '10px', 'border': '1px solid #ddd'}))
        passages.append(html.Div([html.B(f"{block['kind']} · {block['matched_words']} matching words · {block['similarity']:.1%} agreement" + preview_note),
            html.Div(columns, style={'display': 'flex', 'gap': '10px', 'flexWrap': 'wrap'})]))
    return html.Details([html.Summary(
        f"{source['filename']} | {len(source['blocks'])} blocks | "
        f"{source['target_coverage']:.1%} of target / {source['source_coverage']:.1%} of source{notes}"), *passages], open=open_details)


def render_target_passage_search(target_id, min_words, similarity, display_limit, search_data=None):
    if not target_id:
        return 'Select a target document above first.'
    try:
        result, docs = search_data if search_data is not None else run_target_passage_search(target_id, min_words, similarity)
    except (OSError, ValueError, KeyError, TypeError, EOFError) as exc:
        return html.Div(f'Passage search unavailable: {exc}', style={'color': '#a33'})
    target = docs[result['target_id']]
    limit = len(result['sources']) if display_limit == 'all' else int(display_limit)
    shown = result['sources'][:limit]
    children = [html.H4('Passage matches for ' + result['target_filename']),
                html.P(result['target_path']),
                html.P(f"Searched {result['searched_sources']} cached source files; "
                       f"{len(result['sources'])} files with matching passages. "
                       f"Showing {len(shown)}. Minimum {result['min_words']} matched words per block; "
                       f"local word agreement ≥ {result['similarity']:.0%}."),
                html.P('Ranked by matched wording, with common wording downweighted. Coverage counts matching words once, excluding edited words. '
                       'Results are lexical evidence, not proof of a direct copy. Dates do not exclude candidates. '
                       'Five-word anchors can miss densely edited passages. Paragraph numbers are XML locations, not pages. '
                       'This search covers only the collection scan, including its date filters.')]
    if result['skipped']:
        children.append(html.Details([html.Summary(f"{len(result['skipped'])} unavailable, empty, failed or stale files excluded"),
            html.Ul([html.Li(d['filename'] + ': ' + d['reason']) for d in result['skipped']])]))
    if result['limits']:
        children.append(html.Details([html.Summary(f"Search limits affected {len(result['limits'])} files; matches may be incomplete"),
            html.Ul([html.Li(d['filename'] + ': ' + d['reason']) for d in result['limits']])]))
    children.append(render_contributor_groups(result['contributor_groups']))
    children.append(html.H4('Full file-by-file passage evidence'))
    if not shown:
        children.append(html.P('No qualifying wording matches found. This does not prove that no source contributed text.'))
    for source in shown:
        children.append(render_passage_source(source, target, docs[source["source_id"]]))
    return html.Div(children)


# Compact contribution selection; grouping describes shared target passages,
# never an inferred version history. All original passage results are retained.
CONTRIBUTOR_INITIAL_GROUPS = 6
CONTRIBUTOR_SAME_PASSAGES = 0.90
CONTRIBUTOR_WHOLE_TARGET_COVERAGE = 0.85
CONTRIBUTOR_COMPOSITE_EXPLAINED = 0.90


def contributor_intersection(left, right):
    a, b = passage_intervals(left), passage_intervals(right)
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        start, end = max(a[i][0], b[j][0]), min(a[i][1], b[j][1])
        if start < end:
            result.append([start, end])
        if a[i][1] <= b[j][1]:
            i += 1
        else:
            j += 1
    return result


def contributor_difference(spans, covered):
    """Subtract covered positions from supported target spans."""
    result = []
    masks = passage_intervals(covered)
    for start, end in passage_intervals(spans):
        pos = start
        for a, b in masks:
            if b <= pos:
                continue
            if a >= end:
                break
            if a > pos:
                result.append([pos, min(a, end)])
            pos = max(pos, b)
            if pos >= end:
                break
        if pos < end:
            result.append([pos, end])
    return result


def build_contributor_groups(result, docs, lineage_rows=()):
    """Separate broad matches, group equivalents, select complementary sources.

    This is a review heuristic, not attribution of the actual copy operation.
    Complete-link grouping avoids joining different sections through a bridge.
    Coverage and marginal gains always belong to the representative physical file.
    The six-group limit is applied by the renderer only, never by selection.
    """
    sources = result['sources']
    minimum = result['min_words']
    target_words = docs[result['target_id']]['word_count']
    incoming = {}
    links = []
    for row in lineage_rows:
        parent, child = str(row['Parent_ID']), str(row['Child_ID'])
        score = safe_float(row.get('Score', 0))
        links.append({'parent_id': parent, 'child_id': child, 'score': score,
                      'evidence_type': clean_text(row.get('Evidence_Type', 'content'))})
        if child == result['target_id']:
            incoming[parent] = max(incoming.get(parent, 0), score)

    def spans(source):
        return source['target_equal_spans']

    def overlap(a, b):
        return passage_length(contributor_intersection(spans(a), spans(b)))

    def representative_key(source):
        total = max(sum(b['matched_words'] for b in source['blocks']), 1)
        exact = sum(b['matched_words'] for b in source['blocks'] if b['kind'] == 'exact normalized wording')/total
        return (-source['matched_target_words'], bool(source.get('limited')),
                bool(source.get('later_timestamp')), -exact, -incoming.get(source['source_id'], 0),
                docs[source['source_id']]['fingerprint']['mtime_ns'], source['path'], source['source_id'])

    broad = {}
    for source in sources:
        if source['target_coverage'] >= CONTRIBUTOR_WHOLE_TARGET_COVERAGE:
            broad[source['source_id']] = {
                'kind': 'near-whole-document match',
                'reason': f"Matches at least {CONTRIBUTOR_WHOLE_TARGET_COVERAGE:.0%} of the target; possible assembled version or close copy.",
                'explaining_source_ids': []}
            continue
        # A partial assembly can dominate component files too. Separate it only
        # when multiple smaller sources independently explain most of its spans.
        pieces = [other for other in sources if other['source_id'] != source['source_id']
                  and minimum <= other['matched_target_words'] <= source['matched_target_words'] * 0.80
                  and overlap(source, other) >= other['matched_target_words'] * 0.90]
        explained, chosen = [], []
        remaining = pieces[:]
        while remaining:
            best = max(remaining, key=lambda s: (
                passage_length(contributor_difference(contributor_intersection(spans(source), spans(s)), explained)),
                s['matched_target_words'], s['source_id']))
            gain = contributor_difference(contributor_intersection(spans(source), spans(best)), explained)
            if passage_length(gain) < minimum:
                break
            chosen.append(best['source_id'])
            explained = passage_intervals(explained + gain)
            remaining.remove(best)
        if len(chosen) >= 2 and passage_length(explained) >= source['matched_target_words'] * CONTRIBUTOR_COMPOSITE_EXPLAINED:
            broad[source['source_id']] = {
                'kind': 'multi-section match',
                'reason': 'Most matching text is also explained by multiple smaller sources; possible intermediate assembly. Review the alternatives.',
                'explaining_source_ids': chosen}

    def make_groups(items, prefix):
        groups = []
        for source in sorted(items, key=representative_key):
            group = next((g for g in groups if all(
                overlap(source, member) / max(source['matched_target_words'], member['matched_target_words'], 1)
                >= CONTRIBUTOR_SAME_PASSAGES for member in g['members'])), None)
            if group is None:
                group = {'members': []}
                groups.append(group)
            group['members'].append(source)
        for i, group in enumerate(groups, 1):
            members = sorted(group['members'], key=representative_key)
            representative = members[0]
            ids = {s['source_id'] for s in members}
            alternative_spans = passage_intervals(span for s in members for span in spans(s))
            group.update(group_id=f'{prefix}{i:03d}', members=members,
                         representative_id=representative['source_id'],
                         representative=representative,
                         target_equal_spans=spans(representative),
                         member_ids=[s['source_id'] for s in members],
                         alternative_extra_words=passage_length(contributor_difference(alternative_spans, spans(representative))),
                         existing_lineage_links=[link for link in links if link['parent_id'] in ids and link['child_id'] in ids])
        return groups

    broad_groups = make_groups([s for s in sources if s['source_id'] in broad], 'A')
    for group in broad_groups:
        group['classification'] = [dict(source_id=s['source_id'], **broad[s['source_id']]) for s in group['members']]
    component_groups = make_groups([s for s in sources if s['source_id'] not in broad], 'C')
    prefix_weights = [0.0]
    for weight in result.get('target_word_weights', [1.0] * target_words):
        prefix_weights.append(prefix_weights[-1] + weight)

    def weighted_length(spans):
        return sum(prefix_weights[b]-prefix_weights[a] for a, b in spans)

    covered, selected = [], []
    # Common wording is a ranking penalty, not an automatic exclusion: many
    # versions of a real component may share the same wording too.
    remaining = component_groups[:]
    while remaining:
        choices = []
        for group in remaining:
            extra = contributor_difference(group['target_equal_spans'], covered)
            count = passage_length(extra)
            if count >= minimum:
                choices.append((group, extra, count, weighted_length(extra)))
        if not choices:
            break
        group, extra, count, weighted = min(choices, key=lambda choice: (
            -choice[3], -choice[2], representative_key(choice[0]['representative']), choice[0]['group_id']))
        group.update(additional_target_words=count, additional_target_spans=extra,
                     additional_weighted_words=weighted, selection_order=len(selected)+1)
        selected.append(group)
        covered = passage_intervals(covered + group['target_equal_spans'])
        remaining.remove(group)
    selected_ids = {g['group_id'] for g in selected}
    review = []
    for group in component_groups:
        if group['group_id'] not in selected_ids:
            extra = contributor_difference(group['target_equal_spans'], covered)
            group.update(additional_target_words=passage_length(extra), additional_target_spans=extra,
                         review_reason='Mostly common wording; review manually.' if group['representative'].get('common_wording')
                         else f'Adds fewer than {minimum} matching target words beyond the selected groups.')
            review.append(group)
    return {'selected_groups': selected, 'assembly_groups': broad_groups, 'review_groups': review,
            'component_group_count': len(component_groups), 'minimum_additional_words': minimum,
            'covered_target_spans': covered, 'covered_target_words': passage_length(covered),
            'target_coverage': passage_length(covered)/max(target_words, 1),
            'target_word_count': target_words, 'same_passages_threshold': CONTRIBUTOR_SAME_PASSAGES,
            'initial_display_count': CONTRIBUTOR_INITIAL_GROUPS}


def render_contributor_group_card(group, category):
    representative = group['representative']
    header = (f"{representative['filename']} | {representative['target_coverage']:.1%} of target | "
              f"{len(group['members'])} matching file(s)")
    if category == 'selected':
        header = f"{group['selection_order']}. " + header + f" | adds {group['additional_target_words']} words"
    details = [html.P('Representative file: ' + representative['path']),
               html.P('Alternatives share at least 90% of their matched target positions with every other file in this group. '
                      'This groups matching passages; it does not establish a version history. '
                      'Coverage and added words refer to the representative file.')]
    if representative.get('later_timestamp'):
        details.append(html.P('Representative has a later file timestamp than the target; origin is uncertain.'))
    if representative.get('limited'):
        details.append(html.P('Passage search was limited for the representative; evidence may be incomplete.'))
    if representative.get('common_wording'):
        details.append(html.P('This wording occurs in many candidate files and was downweighted. Review whether it is boilerplate or repeated source versions.'))
    if group['alternative_extra_words']:
        details.append(html.P(f"Alternatives match {group['alternative_extra_words']} additional target words outside the representative's matches; inspect them in the full passage report."))
    if category == 'assembly':
        details.extend(html.P(c['kind'] + ': ' + c['reason']) for c in group['classification'])
    if category == 'review':
        details.append(html.P(group['review_reason']))
    details.append(html.Ul([html.Li([
        html.B(member['filename']), include_matching_file_button(member['source_id'], group['group_id']), matching_file_status(member['source_id'], group['group_id']),
        f" | ID {member['source_id']} | {member['target_coverage']:.1%} of target | "
        + ('later timestamp; origin uncertain | ' if member.get('later_timestamp') else '')
        + ('search limited | ' if member.get('limited') else ''),
        html.Span(member['path']),
        html.Details([html.Summary('Matching target locations'), html.Ul([
            html.Li(f"XML paragraphs {block['target_paragraphs']}; words [{block['target_start']}, {block['target_end']}); {block['matched_words']} matching words")
            for block in member['blocks']])]),
    ]) for member in group['members']]))
    if group['existing_lineage_links']:
        details.append(html.P('Existing lineage evidence within this group: ' + '; '.join(
            f"{link['parent_id']} → {link['child_id']} ({link['evidence_type']}, score {link['score']:.3f})"
            for link in group['existing_lineage_links'])))
    return html.Details([html.Summary(header), *details])


def render_contributor_groups(grouping):
    selected = grouping['selected_groups']
    initial = grouping['initial_display_count']
    children = [html.H4('Compact contributor groups'),
                html.P(f"{len(selected)} complementary groups selected; together their representatives match "
                       f"{grouping['covered_target_words']} words ({grouping['target_coverage']:.1%} of target). "
                       f"Each selection adds at least {grouping['minimum_additional_words']} matching words. "
                       'Counts depend on selection order; they are not exclusive source attribution.'),
                html.P('Near-whole-document and multi-section matches are shown separately so possible assembled versions do not suppress component sources. '
                       'These classifications are text-based review hints. Expand groups to inspect alternative files and their target locations. '
                       'Full side-by-side passage comparisons follow below; choose All matches to inspect every alternative.')]
    if not selected:
        children.append(html.P('No complementary component groups qualified. Review assembled-version matches and the remaining candidates below.'))
    children.extend(render_contributor_group_card(g, 'selected') for g in selected[:initial])
    if len(selected) > initial:
        children.append(html.Details([html.Summary(f'Show {len(selected)-initial} more qualifying contributor groups'),
            *[render_contributor_group_card(g, 'selected') for g in selected[initial:]]]))
    if grouping['assembly_groups']:
        children.append(html.Details([html.Summary(f"Possible assembled versions / broad matches ({len(grouping['assembly_groups'])} groups)"),
            *[render_contributor_group_card(g, 'assembly') for g in grouping['assembly_groups']]]))
    if grouping['review_groups']:
        children.append(html.Details([html.Summary(f"Other matching groups to review ({len(grouping['review_groups'])})"),
            *[render_contributor_group_card(g, 'review') for g in grouping['review_groups']]]))
    return html.Div(children)


# Search results stay on the server. Browser state contains only a random token.
_contributor_sessions = OrderedDict()
_contributor_session_lock = Lock()
CONTRIBUTOR_SESSION_LIMIT = 6


def save_contributor_session(result, docs):
    token = uuid.uuid4().hex
    stat = os.stat(PASSAGE_CACHE_FILE)
    if (stat.st_size, stat.st_mtime_ns) != result['cache_signature']:
        raise ValueError('Text cache changed during search. Repeat the search after restarting the dashboard.')
    entry = {'result': result, 'docs': docs, 'cache_signature': (stat.st_size, stat.st_mtime_ns)}
    with _contributor_session_lock:
        _contributor_sessions[token] = entry
        while len(_contributor_sessions) > CONTRIBUTOR_SESSION_LIMIT:
            _contributor_sessions.popitem(last=False)
    return {'token': token, 'target_id': result['target_id']}


def current_contributor_session(state, target_id):
    if not state or state.get('target_id') != str(target_id):
        return None
    with _contributor_session_lock:
        entry = _contributor_sessions.get(state.get('token'))
    if entry is None:
        raise ValueError('Contributor search expired or dashboard restarted. Click Find text contributors again.')
    if entry['result']['target_id'] != str(target_id):
        return None
    stat = os.stat(PASSAGE_CACHE_FILE)
    if (stat.st_size, stat.st_mtime_ns) != entry['cache_signature']:
        raise ValueError('Text cache changed. Restart the dashboard after collection and repeat the search.')
    check_contributor_document(entry['docs'][str(target_id)])
    return entry


def check_contributor_document(doc):
    stat = os.stat(doc['path'])
    if (stat.st_size, stat.st_mtime_ns) != (doc['fingerprint']['size_bytes'], doc['fingerprint']['mtime_ns']):
        raise ValueError(doc['filename'] + ' changed since collection; regenerate the cache and repeat the search.')


def block_agreement_summary(blocks):
    """Aggregate local word agreement, separately from whole-target coverage.

    Overall = matching words summed across blocks / compared words summed
    across blocks. Compared words = max(source span length, target span length).
    Thus longer blocks carry more weight. Lowest is the weakest individual
    block agreement. Overlapping blocks retain their own comparison weights;
    target coverage continues to use the union of matching target positions.
    These measures compare normalized wording, not meaning or formatting.
    """
    valid = []
    for block in blocks:
        compared = max(block['source_end'] - block['source_start'],
                       block['target_end'] - block['target_start'])
        if compared > 0:
            valid.append((block, compared))
    total = sum(compared for _, compared in valid)
    return {
        'overall': sum(block['matched_words'] for block, _ in valid)/total if total else None,
        'lowest': min(block['similarity'] for block, _ in valid) if valid else None,
        'words_compared': total,
    }


def block_agreement_label(summary):
    if summary['overall'] is None or summary['lowest'] is None:
        return 'Block agreement: unavailable'
    return f"Block agreement: overall {summary['overall']:.1%} / lowest {summary['lowest']:.1%}"


def plan_contributor_overlay(base_graph, view_graph, state, target_id, depth, options, group_limit, assembly_ids=None, manual_ids=None):
    if 'show' not in (options or []):
        return set(), [], ''
    try:
        entry = current_contributor_session(state, target_id)
    except (OSError, ValueError) as exc:
        return set(), [], 'Contributor overlay unavailable: ' + str(exc)
    if entry is None:
        return set(), [], 'Click Find text contributors for this target to add contributor branches.'
    groups = entry['result']['contributor_groups']['selected_groups']
    shown = groups if group_limit == 'all' else groups[:max(1, int(group_limit or 6))]
    classified = [(group, 'component') for group in shown]
    assembly_groups = entry['result']['contributor_groups']['assembly_groups']
    if 'assemblies' in options:
        classified += [(g, 'intermediate') for g in assembly_groups if not assembly_ids or g['group_id'] in assembly_ids]
    for source in entry['result']['sources']:
        if source['source_id'] in (manual_ids or []):
            classified.append(({'group_id': 'M'+source['source_id'], 'members': [source],
                                'representative': source, 'representative_id': source['source_id'],
                                'additional_target_words': None}, 'manual'))
    extra, contributions, omitted = set(), [], []
    physical_edges = {}
    target = base_graph['original_to_group'].get(str(target_id), str(target_id))
    for group, role in classified:
        members = group['members'] if 'alternatives' in options else [group['representative']]
        # Under duplicate collapse, keep the representative's physical evidence.
        members = sorted(members, key=lambda s: (s['source_id'] != group['representative_id'], s['source_id']))
        seen = set()
        for source in members:
            physical_id = source['source_id']
            logical_id = base_graph['original_to_group'].get(physical_id, physical_id)
            if physical_id in physical_edges:
                if role == 'manual':
                    physical_edges[physical_id]['data']['manually_included'] = True
                continue
            if logical_id in seen:
                continue
            try:
                check_contributor_document(entry['docs'][physical_id])
            except (OSError, ValueError) as exc:
                omitted.append(str(exc))
                continue
            seen.add(logical_id)
            if logical_id == target:
                omitted.append(source['filename'] + ': collapsed into the target; inspect physical evidence in the report.')
                continue
            extra.add(logical_id)
            if 'history' in options:
                extra |= traverse(logical_id, view_graph['parents'], int(depth))
            alternative = physical_id != group['representative_id']
            label = ('Intermediate text match' if role == 'intermediate' else
                     'Manually included text match' if role == 'manual' else
                     'Alternative text match' if alternative else 'Text contribution')
            label += f"\n{len(source['blocks'])} blocks · {source['target_coverage']:.1%} of target"
            agreement = block_agreement_summary(source['blocks'])
            label += '\n' + block_agreement_label(agreement)
            if source['later_timestamp']:
                label += '\nLater timestamp; origin uncertain'
            if source['limited']:
                label += '\nSearch limited'
            contributions.append({'data': {
                'id': f"contribution-{state['token']}-{group['group_id']}-{physical_id}",
                'source': logical_id, 'target': target, 'evidence_type': 'text_contribution',
                'evidence_parent_id': physical_id, 'evidence_child_id': str(target_id),
                'contributor_group_id': group['group_id'], 'label': label, 'overlay_role': role, 'manually_included': role == 'manual',
                'relationship': 'matching wording; possible direct or indirect contribution',
                'matched_target_words': source['matched_target_words'], 'target_coverage': source['target_coverage'],
                'block_count': len(source['blocks']), 'alternative': alternative,
                'overall_block_agreement': agreement['overall'], 'lowest_block_agreement': agreement['lowest'],
                'block_agreement_words_compared': agreement['words_compared'],
                'additional_target_words': None if alternative or role != 'component' else group['additional_target_words'],
                'later_timestamp': source['later_timestamp'], 'search_limited': source['limited'],
                'physical_source_filename': source['filename'],
                'physical_target_filename': entry['result']['target_filename'],
            }, 'classes': 'contribution-edge' + (' contribution-alternative' if alternative else '') + (' intermediate-edge' if role == 'intermediate' else '') + (' manual-match-edge' if role == 'manual' else '')})
            physical_edges[physical_id] = contributions[-1]
    note = (f'Contributor overlay: {len(shown)} of {len(groups)} selected groups; {len(contributions)} text-match arrows. '
            'Green dashed arrows show matching wording, possibly incorporated through an intermediate file; they do not prove direct copying. '
            'History branches follow the current depth, score and parent filters. Contribution arrows bypass those filters and transitive-edge hiding. ')
    if len(shown) < len(groups):
        note += 'Choose All component groups to reveal the remaining component contributions. '
    note += 'Purple text-match arrows show intermediate assemblies; blue arrows show manually included matches. Assemblies have separate controls. '
    if omitted:
        note += 'Omitted matches: ' + '; '.join(omitted)
    return extra, contributions, note


def decorate_contributor_elements(elements, contributions):
    if not contributions:
        return elements
    source_ids = {edge['data']['source'] for edge in contributions}
    for item in elements:
        data = item['data']
        if 'source' in data:
            data['id'] = f"history-{data['source']}-{data['target']}"
        elif data.get('id') in source_ids:
            item['classes'] = item.get('classes', '') + ' text-contributor'
            matches = [edge['data'] for edge in contributions if edge['data']['source'] == data['id']]
            data['focus_original_id'] = matches[0]['evidence_parent_id']
            data['contributor_physical_ids'] = ' | '.join(edge['evidence_parent_id'] for edge in matches)
    return elements + contributions


def render_contributor_edge(data, elements, state, target_id):
    if not data or not any(e.get('data', {}).get('id') == data.get('id') for e in (elements or [])):
        return 'Choose a contribution arrow in the current view.'
    try:
        entry = current_contributor_session(state, target_id)
        if entry is None or str(target_id) != data.get('evidence_child_id'):
            return 'Contributor evidence belongs to another target. Repeat the search.'
        source = next(s for s in entry['result']['sources'] if s['source_id'] == data.get('evidence_parent_id'))
        doc = entry['docs'][source['source_id']]
        target = entry['docs'][str(target_id)]
        check_contributor_document(doc)
        agreement = block_agreement_summary(source['blocks'])
        return html.Div([html.H4('Matching contributor passages'),
            html.P(block_agreement_label(agreement)),
            html.P('Overall agreement is weighted by compared block length; lowest is the weakest detected block. Target coverage measures matching text as a share of the whole target.'),
            html.P('Displayed as: ' + data.get('overlay_role', 'component') + ' match'),
            html.P('Physical source: ' + doc['path']), html.P('Physical target: ' + target['path']),
            html.P('This arrow identifies shared wording, possibly through an intermediate document. '
                   'Locations and coverage belong to these physical files, including when their nodes are grouped.'),
            render_passage_source(source, target, doc, open_details=True, include_context='edge')])
    except (OSError, ValueError, KeyError, StopIteration) as exc:
        return html.Div('Contributor evidence unavailable: ' + str(exc), style={'color': '#a33'})


def include_matching_file_button(source_id, context):
    return html.Button('Show this file in the graph',
        id={'type': 'include-matching-file', 'source_id': source_id, 'context': context}, n_clicks=0,
        style={'margin': '6px', 'padding': '6px'})


def matching_file_status(source_id, context):
    return html.Div(id={'type': 'matching-file-status', 'source_id': source_id, 'context': context},
                    children='Graph status will update with the current view.')


app = dash.Dash(__name__)
app.title = 'DOCX Lineage Explorer'
app.layout = html.Div([html.H2('DOCX Lineage Explorer'), html.Label('Target document — select your merged file here'), dcc.Dropdown(id='document', options=document_options, placeholder='Search filename or path...', searchable=True), html.Br(), html.Div([html.Div([html.Label('Direction'), dcc.Dropdown(id='direction', options=[{'label': 'Ancestors + descendants', 'value': 'both'}, {'label': 'Ancestors only', 'value': 'ancestors'}, {'label': 'Descendants only', 'value': 'descendants'}], value=DEFAULT_DIRECTION, clearable=False)], style={'width': '18%', 'display': 'inline-block', 'paddingRight': '14px', 'verticalAlign': 'top'}), html.Div([html.Label('Generation depth'), dcc.Slider(id='depth', min=1, max=10, step=1, value=DEFAULT_DEPTH, marks={i: str(i) for i in range(1, 11)})], style={'width': '18%', 'display': 'inline-block', 'paddingRight': '14px', 'verticalAlign': 'top'}), html.Div([html.Label('Minimum content score'), dcc.Slider(id='min-score', min=0, max=1, step=0.05, value=DEFAULT_MIN_SCORE, marks={0: '0', 0.25: '.25', 0.5: '.50', 0.75: '.75', 1: '1'})], style={'width': '18%', 'display': 'inline-block', 'paddingRight': '14px', 'verticalAlign': 'top'}), html.Div([html.Label('Layout'), dcc.RadioItems(id='layout-mode', options=[{'label': ' Lineage', 'value': 'lineage'}, {'label': ' Time sequence', 'value': 'time'}], value=DEFAULT_LAYOUT, labelStyle={'display': 'block', 'marginBottom': '6px'})], style={'width': '15%', 'display': 'inline-block', 'paddingRight': '14px', 'verticalAlign': 'top'}), html.Div([html.Label('Duplicate grouping'), dcc.Dropdown(id='collapse-mode', options=[{'label': 'Off', 'value': 'off'}, {'label': 'Exact binary copies', 'value': 'binary'}, {'label': 'Same normalized text (aggressive)', 'value': 'text'}], value=DEFAULT_COLLAPSE_MODE, clearable=False)], style={'width': '23%', 'display': 'inline-block', 'verticalAlign': 'top'})]), html.Br(), html.Div([html.Div([html.Label('Graph view'), dcc.RadioItems(id='view-mode', options=[{'label': ' Likely history', 'value': 'likely'}, {'label': ' Full evidence graph', 'value': 'evidence'}], value=DEFAULT_VIEW_MODE, labelStyle={'display': 'block', 'marginBottom': '5px'})], style={'width': '18%', 'display': 'inline-block', 'paddingRight': '18px', 'verticalAlign': 'top'}), html.Div([html.Label('Edited/inferred parents per document (+1 copy)'), dcc.Dropdown(id='parents-per-node', options=[{'label': '1', 'value': 1}, {'label': '2', 'value': 2}, {'label': '3', 'value': 3}], value=DEFAULT_PARENTS_PER_NODE, clearable=False)], style={'width': '18%', 'display': 'inline-block', 'paddingRight': '18px', 'verticalAlign': 'top'}), html.Div([html.Label('Simplify edges'), dcc.Checklist(id='edge-options', options=[{'label': ' Show inferred candidate edges (separate from content filter)', 'value': 'candidates'}, {'label': ' Hide transitive/redundant edges', 'value': 'transitive'}, {'label': ' Show detailed diff on edge labels', 'value': 'details'}, {'label': ' Include filename-related files (may be unlinked)', 'value': 'filename-relatives'}], value=['transitive', 'details', 'candidates'], labelStyle={'display': 'block', 'marginBottom': '5px'})], style={'width': '30%', 'display': 'inline-block', 'paddingRight': '18px', 'verticalAlign': 'top'}), html.Div([html.Button('Use clicked node as focus', id='focus-clicked-button', n_clicks=0, style={'marginTop': '22px', 'padding': '8px 12px'})], style={'width': '25%', 'display': 'inline-block', 'verticalAlign': 'top'})]), html.Br(), html.Label('Time spacing'), dcc.RadioItems(id='time-spacing', options=[{'label': ' Compact chronological order', 'value': 'compact'}, {'label': ' Proportional elapsed time', 'value': 'proportional'}], value=DEFAULT_TIME_SPACING, inline=True), html.Br(), dcc.Checklist(id='branch-colors', options=[{'label': ' Color branches in Likely history', 'value': 'branches'}], value=['branches'] if DEFAULT_BRANCH_COLORS else []), html.Hr(), html.Div(id='network-info'), cyto.Cytoscape(id='lineage-network', elements=[], layout={'name': 'preset', 'fit': True, 'padding': 60}, style={'width': '100%', 'height': '72vh', 'minHeight': '650px', 'border': '1px solid #cccccc'}, stylesheet=[{'selector': 'node', 'style': {'label': 'data(label)', 'font-size': NODE_FONT_SIZE, 'text-wrap': 'wrap', 'text-max-width': LABEL_MAX_WIDTH, 'text-overflow-wrap': 'anywhere', 'width': '36px', 'height': '36px', 'background-color': 'data(branch_color)'}}, {'selector': '.duplicate-group', 'style': {'shape': 'round-rectangle', 'border-width': '3px', 'border-color': '#557a95', 'width': '45px', 'height': '45px'}}, {'selector': '.filename-relative', 'style': {'border-width': '3px', 'border-style': 'dashed', 'border-color': '#806b91'}}, {'selector': '.selected-document', 'style': {'background-color': '#e84a5f', 'width': '60px', 'height': '60px', 'border-width': '4px', 'border-color': '#333333', 'font-size': SELECTED_FONT_SIZE, 'font-weight': 'bold'}}, {'selector': 'edge', 'style': {'curve-style': 'bezier', 'target-arrow-shape': 'triangle', 'label': 'data(label)', 'font-size': EDGE_FONT_SIZE, 'text-wrap': 'wrap', 'line-color': '#aaaaaa', 'target-arrow-color': '#777777', 'width': 2}}, {'selector': '.candidate-edge', 'style': {'line-style': 'dashed', 'line-color': '#806b91', 'target-arrow-color': '#806b91'}}, {'selector': '.chronology-ambiguous', 'style': {'line-style': 'dashed', 'line-color': '#d59b32', 'target-arrow-color': '#d59b32'}}, {'selector': '.chronology-conflict', 'style': {'line-style': 'dashed', 'line-color': '#cc4c4c', 'target-arrow-color': '#cc4c4c'}}]), html.Hr(), html.H3('Clicked edge passages'), dcc.Loading(html.Div(id='edge-passage-comparison', children='Click an edge to compare its source (A) and target (B).'), type='default'), html.H3('Clicked edge details'), html.Div(id='edge-details', style={'padding': '10px', 'border': '1px solid #dddddd', 'whiteSpace': 'pre-wrap', 'fontFamily': 'monospace'}), html.Hr(), html.H3('Likely latest versions'), html.P('A shortlist for review, not a declaration of finality. Uses all loaded CSV evidence, independently of the graph filters. Equal normalized text is grouped only in this table; case, punctuation, formatting, images and comments may still differ.'), html.P('Text first file time is the earliest modification timestamp among matching files in this scan, not the actual creation date of the revision. Newer copies do not advance this date. Separate branches are retained.'), dcc.RadioItems(id='latest-scope', options=[{'label': ' Selected family and linked documents', 'value': 'related'}, {'label': ' All document families', 'value': 'all'}], value='related', inline=True), dcc.Checklist(id='latest-options', options=[{'label': ' Also show versions with a later linked text revision', 'value': 'previous'}], value=[]), html.Div(id='latest-versions-table'), html.Hr(), html.H3('Compare two files'), html.P('Choose any two physical files from the loaded scan. Compare reads their current contents from disk only when clicked; no graph connection is required. A is before, B is after. Existing results identify the files actually compared.'), html.P('Compares main-body and table text, preserving case and punctuation and normalizing whitespace. Formatting, images, headers, footers, footnotes and comments are not compared.'), html.Div([html.Div([html.Label('A ? Before'), dcc.Dropdown(id='compare-before', options=document_options, searchable=True, placeholder='Search filename or path...')], style={'flex':'1','minWidth':'300px'}), html.Div([html.Label('B ? After'), dcc.Dropdown(id='compare-after', options=document_options, searchable=True, placeholder='Search filename or path...')], style={'flex':'1','minWidth':'300px'})], style={'display':'flex','gap':'14px','flexWrap':'wrap'}), html.Br(), html.Button('Compare selected files', id='compare-files-button', n_clicks=0), dcc.Loading(html.Div(id='compare-files-result', children='Select two physical files and click Compare.', style={'marginTop':'12px'}), type='default'), html.Hr(), html.H3('Likely immediate predecessors'), html.P('This table always uses all incoming evidence for the selected document, even when the graph itself is simplified.'), html.Div(id='predecessor-table'), html.Br(), html.Button('Download predecessor report (.txt)', id='download-report-button', n_clicks=0), dcc.Download(id='download-report'), html.Hr(), html.H3('Selected node / duplicate group'), html.Div(id='node-details', style={'padding': '10px', 'border': '1px solid #dddddd'})], style={'maxWidth': '1900px', 'margin': '0 auto', 'padding': '14px'})

@app.callback(Output('lineage-network', 'elements'), Output('lineage-network', 'layout'), Output('network-info', 'children'), Output('predecessor-table', 'children'), Input('document', 'value'), Input('direction', 'value'), Input('depth', 'value'), Input('min-score', 'value'), Input('layout-mode', 'value'), Input('collapse-mode', 'value'), Input('view-mode', 'value'), Input('parents-per-node', 'value'), Input('edge-options', 'value'), Input('branch-colors', 'value'), Input('time-spacing', 'value'), Input('contributor-session', 'data'), Input('contributor-overlay-options', 'value'), Input('contributor-group-limit', 'value'), Input('show-modification-dates', 'value'), Input('intermediate-assemblies', 'value'), Input('manual-matching-files', 'value'))
def update_network(selected_original_id, direction, depth, min_score, layout_mode, collapse_mode, view_mode, parents_per_node, edge_options, branch_options=None, time_spacing='compact', contributor_state=None, contributor_options=None, contributor_limit='6', date_options=None, assembly_ids=None, manual_ids=None):
    if not selected_original_id:
        return ([], {'name': 'preset'}, 'Choose a document above.', '')
    base_graph = build_graph_data(collapse_mode)
    selected_id = base_graph['original_to_group'].get(selected_original_id, selected_original_id)
    hide_transitive = 'transitive' in (edge_options or [])
    show_detail_labels = 'details' in (edge_options or [])
    view_graph = cached_view_graph(collapse_mode, view_mode, parents_per_node, min_score, hide_transitive, 'candidates' in (edge_options or []))
    color_branches = view_mode == 'likely' and 'branches' in (branch_options or [])
    extra_nodes, contribution_edges, contributor_note = plan_contributor_overlay(base_graph, view_graph, contributor_state, selected_original_id, depth, contributor_options, contributor_limit, assembly_ids, manual_ids)
    elements = build_elements(view_graph, selected_id, direction, depth, layout_mode, show_detail_labels, color_branches, time_spacing, 'filename-relatives' in (edge_options or []), extra_nodes=extra_nodes, show_dates=None if date_options is None else 'dates' in date_options)
    elements = decorate_contributor_elements(elements, contribution_edges)
    n_nodes = sum((1 for e in elements if 'id' in e.get('data', {}) and 'source' not in e['data']))
    n_edges = sum((1 for e in elements if 'source' in e.get('data', {})))
    if layout_mode == 'time':
        layout = {'name': 'preset', 'fit': True, 'padding': 60}
        layout_description = ('Compact sequence runs left to right, including separate copies. Equal-time files are ordered by ID for display only, not proven save order; gaps do not represent elapsed time.' if time_spacing == 'compact' else 'Proportional time sequence uses the earliest Windows Modified time for each logical document state.')
    else:
        layout = {'name': 'breadthfirst', 'directed': True, 'padding': 50, 'spacingFactor': TREE_SPACING_FACTOR, 'nodeDimensionsIncludeLabels': True}
        layout_description = 'Lineage layout emphasizes parent/child structure.'
    selected_group = base_graph['nodes'].get(selected_id, {})
    hidden_duplicates = len(nodes) - len(base_graph['nodes'])
    mode_description = f'Likely history keeps at most {parents_per_node} edited/inferred predecessor(s), plus one same-text copy, per document.' if view_mode == 'likely' else 'Full evidence graph keeps all edges above the score threshold.'
    collapse_description = {'off': 'Duplicate grouping is off.', 'binary': f'Exact binary copies collapsed: {hidden_duplicates} physical node(s) hidden.', 'text': f'Same normalized-text copies collapsed: {hidden_duplicates} physical node(s) hidden.'}.get(collapse_mode, '')
    hint_count = sum(bool(e['data'].get('filename_hint')) for e in elements)
    hidden_relatives = filename_relatives(base_graph, selected_id) - {e['data']['id'] for e in elements if 'id' in e['data']}
    visibility_note = (f'{hint_count} extra filename-related node(s) have dashed borders; filename matches do not establish lineage. ' if hint_count else '')
    if hidden_relatives:
        names = sorted({base_graph['nodes'][nid]['representative_filename'] for nid in hidden_relatives})
        visibility_note += 'Filename-related files outside this view: ' + ', '.join(names) + '. Enable Include filename-related files, or select one as focus. '
    if direction == 'ancestors':
        visibility_note += 'Ancestors only excludes later versions unless filename-related files are included.'
    info = html.Div([html.B(f'Showing {n_nodes} logical documents and {n_edges} relationships. '), html.Span(f"Selected logical node contains {selected_group.get('copy_count', 1)} physical file(s). {collapse_description} {mode_description} {layout_description}"), html.Br(), html.Span('Purple dashed edges = inferred candidates (not proven ancestry; candidate rank is not a probability). The content filter applies only to content edges. Amber dashed edges = autosave-time ambiguity; red dashed edges = stronger chronology conflict. A is the edge source; B is the target. Content is broad similarity. Shared text percentages estimate overlap using distinct word shingles in A and B, not proof of origin. Click an edge for its current passages side by side. Same extracted text may still differ in formatting or images.'), html.Br(), html.Span('Branch colors follow splits in the displayed likely history; merges and ambiguous structure stay blue. Colors may repeat and are not proof of ancestry. Selected document = red.' if color_branches else 'Branch colors are off (available in Likely history).'), html.Br(), html.Span(visibility_note)])
    if contributor_note:
        info.children.extend([html.Br(), html.Span(contributor_note)])
    table = predecessor_table(base_graph, selected_id, min_score=0.0)
    if table.empty:
        table_component = html.Div('No direct incoming candidate parents were found.')
    else:
        display_table = table.head(15).copy()
        display_table['Rank_Score'] = display_table['Rank_Score'].map(lambda x: f'{x:.3f}')
        display_table['Content_Score'] = display_table['Content_Score'].map(lambda x: f'{x:.3f}')
        display_table['Word_Delta_Pct'] = display_table['Word_Delta_Pct'].map(lambda x: '' if pd.isna(x) else f'{x:+.2f}%')
        display_table['Text_Delta_Pct'] = display_table['Text_Delta_Pct'].map(lambda x: '' if x == '' or pd.isna(x) else f'{float(x):.4f}%')
        display_table['Diff_Edit_Blocks'] = display_table['Diff_Edit_Blocks'].map(lambda x: '' if x == '' or pd.isna(x) else str(safe_int(x)))
        display_table['Changed_Paragraphs_Est'] = display_table['Changed_Paragraphs_Est'].map(lambda x: '' if x == '' or pd.isna(x) else str(safe_int(x)))
        cols = ['Parent_File', 'Evidence_Type', 'Candidate_Score', 'Copies', 'Parent_Modified', 'Time_To_Child', 'Chronology', 'Relationship', 'Content_Score', 'Text_Delta_Pct', 'Diff_Edit_Blocks', 'Changed_Paragraphs_Est', 'Diff_Class', 'Rank_Score', 'Word_Delta_Pct']
        table_component = html.Div([html.Table([html.Thead(html.Tr([html.Th(c, style={'padding': '6px 8px', 'textAlign': 'left', 'whiteSpace': 'nowrap'}) for c in cols])), html.Tbody([html.Tr([html.Td(str(row[c]), style={'padding': '5px 8px', 'borderBottom': '1px solid #ddd', 'verticalAlign': 'top'}) for c in cols]) for _, row in display_table.iterrows()])], style={'borderCollapse': 'collapse', 'width': '100%', 'fontSize': '12px'})], style={'overflowX': 'auto'})
    return (elements, layout, info, table_component)

@app.callback(Output('node-details', 'children'), Input('lineage-network', 'tapNodeData'))
def show_node_details(data):
    if not data:
        return 'Click a document node to inspect it.'
    paths = clean_text(data.get('paths', '')).splitlines()
    path_items = []
    for i, path in enumerate(paths, start=1):
        path_items.append(html.Div([html.Div(f'{i}.', style={'fontWeight': 'bold', 'marginRight': '8px'}), dcc.Input(value=path, type='text', readOnly=True, style={'width': '95%', 'fontFamily': 'monospace'})], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '5px'}))
    return html.Div([html.Div([html.B('Logical node: '), data.get('filename', '')]), html.Div([html.B('Filenames in this group: '), data.get('filenames', '')], style={'whiteSpace': 'pre-wrap'}), html.Div('Added by filename match; this alone does not establish a lineage relationship.' if data.get('filename_hint') else ''), html.Div([html.B('Logical ID: '), data.get('id', '')]), html.Div([html.B('Physical copies: '), data.get('copy_count', 1)]), html.Div([html.B('Member IDs: '), data.get('member_ids', '')]), html.Div([html.B('Earliest modified: '), data.get('modified', '')]), html.Div([html.B('Latest modified: '), data.get('modified_latest', '')]), html.Div([html.B('Words: '), data.get('words', '')]), html.Div([html.B('Revision(s): '), data.get('revision', '')]), html.Div([html.B('Creator(s): '), data.get('creator', '')]), html.Br(), html.B('Physical locations (click field, Ctrl+A, Ctrl+C):'), html.Div(path_items, style={'marginTop': '6px'})])

@app.callback(Output('edge-passage-comparison', 'children'), Input('lineage-network', 'tapEdgeData'), Input('lineage-network', 'elements'), State('contributor-session', 'data'), State('document', 'value'))
def show_edge_passages(data, elements, contributor_state=None, target_id=None):
    if 'lineage-network.elements' in dash.ctx.triggered_prop_ids:
        return 'Click an edge in this view to compare its source (A) and target (B). '
    if data and data.get('evidence_type') == 'text_contribution':
        return render_contributor_edge(data, elements, contributor_state, target_id)
    return compare_edge_files(data, elements)


@app.callback(Output('edge-details', 'children'), Input('lineage-network', 'tapEdgeData'), Input('lineage-network', 'elements'))
def show_edge_details(data, elements=None):
    if elements is not None and 'lineage-network.elements' in dash.ctx.triggered_prop_ids:
        return 'Click an edge in the current view to inspect its evidence.'
    if not data:
        return 'Click an edge to inspect its similarity and detailed-diff evidence.'
    if data.get('evidence_type') == 'text_contribution':
        return '\n'.join([
            'Physical source: ' + data.get('physical_source_filename', ''),
            'Physical target: ' + data.get('physical_target_filename', ''),
            'Group: ' + data.get('contributor_group_id', ''),
            f"Matching blocks: {data.get('block_count', 0)}",
            f"Matching target words: {data.get('matched_target_words', 0)}",
            f"Target coverage: {data.get('target_coverage', 0):.1%}",
            block_agreement_label({'overall': data.get('overall_block_agreement'), 'lowest': data.get('lowest_block_agreement')}),
            'Overall agreement is weighted by compared block length; lowest is the weakest detected block. Both compare normalized wording, not whole-file similarity.',
            'Alternative file: ' + str(data.get('alternative', False)),
            'Later timestamp: ' + str(data.get('later_timestamp', False)),
            'Possible direct or indirect text contribution; not proven copy provenance.',
        ])
    lines = [f"Source:                 {data.get('source', '')}", f"Target:                 {data.get('target', '')}", f"Relationship:           {data.get('relationship', '')}", f"Broad content score:    {data.get('score', '')}", f"Time gap:               {data.get('gap', '')}", f"Chronology:             {data.get('chronology', '')}", f"Aggregated evidence:    {data.get('evidence_count', '')}", '', f"Detailed diff run:      {data.get('detail_diff_run', '')}", f"Exact visible text:     {data.get('exact_visible_equal', '')}", f"Diff class:             {data.get('diff_class', '')}", f"Text delta %:           {data.get('text_delta_pct', '')}", f"Token similarity:       {data.get('token_similarity', '')}", f"Diff edit blocks:       {data.get('diff_edit_blocks', '')}", f"Changed paragraphs est: {data.get('changed_paragraphs', '')}", f"Changed tokens est:     {data.get('changed_tokens', '')}", f"Inserted tokens:        {data.get('inserted_tokens', '')}", f"Deleted tokens:         {data.get('deleted_tokens', '')}", f"Replaced old tokens:    {data.get('replaced_old_tokens', '')}", f"Replaced new tokens:    {data.get('replaced_new_tokens', '')}", '', f"Old content in new:     {data.get('old_content_in_new', '')}", f"New content in old:     {data.get('new_content_in_old', '')}", f"Paragraph Jaccard:      {data.get('paragraph_jaccard', '')}", f"Shingle Jaccard:        {data.get('shingle_jaccard', '')}"]
    lines.extend(['', 'Evidence type: ' + clean_text(data.get('evidence_type', 'content')), 'Candidate rank (not probability): ' + clean_text(data.get('candidate_score', '')), 'Candidate evidence: ' + clean_text(data.get('candidate_reason', '')), 'Version progression: ' + clean_text(data.get('version_progression', '')), 'Filename date order: ' + clean_text(data.get('filename_date_order', '')), 'Old shingles in new: ' + clean_text(data.get('old_shingle_containment', '')), 'New shingles in old: ' + clean_text(data.get('new_shingle_containment', ''))])
    if data.get('changed_old_sample') or data.get('changed_new_sample'):
        lines.extend(['', 'Sample from older file:', clean_text(data.get('changed_old_sample', '')), '', 'Sample from newer file:', clean_text(data.get('changed_new_sample', ''))])
    return '\n'.join(lines)

@app.callback(Output('document', 'value'), Input('focus-clicked-button', 'n_clicks'), State('lineage-network', 'tapNodeData'), prevent_initial_call=True)
def focus_clicked_node(n_clicks, node_data):
    if not node_data:
        return dash.no_update
    node_id = clean_text(node_data.get('focus_original_id', node_data.get('id', '')))
    return node_id if node_id else dash.no_update

@app.callback(Output('download-report', 'data'), Input('download-report-button', 'n_clicks'), State('document', 'value'), State('collapse-mode', 'value'), prevent_initial_call=True)
def download_predecessor_report(n_clicks, selected_original_id, collapse_mode):
    if not selected_original_id:
        return None
    base_graph = build_graph_data(collapse_mode)
    selected_id = base_graph['original_to_group'].get(selected_original_id, selected_original_id)
    report = predecessor_report_text(base_graph, selected_id, min_score=0.0)
    filename = base_graph['nodes'].get(selected_id, {}).get('representative_filename', selected_id)
    safe_name = ''.join((c if c.isalnum() or c in '._-' else '_' for c in str(filename)))[:100]
    return dict(content=report, filename=f'{safe_name}_predecessor_report.txt')
@app.callback(Output('latest-versions-table', 'children'), Input('document', 'value'), Input('latest-scope', 'value'), Input('latest-options', 'value'))
def update_latest_versions(selected_original_id, scope, options):
    return render_latest_versions(selected_original_id, scope, options)

@app.callback(Output('compare-files-result', 'children'), Input('compare-files-button', 'n_clicks'), State('compare-before', 'value'), State('compare-after', 'value'), prevent_initial_call=True)
def run_file_comparison(n_clicks, old_id, new_id):
    return compare_selected_files(n_clicks, old_id, new_id)


# Optional analysis runs only on button clicks, using the selected physical file.
app.layout.children[4:4] = [dcc.Checklist(id='show-modification-dates', options=[{'label': ' Show modification date/time', 'value': 'dates'}], value=['dates']), dcc.Store(id='contributor-session', storage_type='memory'), html.Details([
    html.Summary('Find source files for the selected target'),
    html.Div(id='contributor-target-label'),
    html.P('1. Select your merged target in the file selector above. 2. Click Find text contributors below. '
           'The report includes compact contributor groups, possible assembled versions, and all matching-file evidence. Selected contributor branches are added to the tree after searching.'),
    html.Div([
        html.Label('Minimum matching words per block '),
        dcc.Input(id='passage-min-words', type='number', min=5, max=500, step=1, value=PASSAGE_MIN_WORDS),
        html.Label(' Minimum local word agreement (0.5–1) '),
        dcc.Input(id='passage-similarity', type='number', min=0.5, max=1, step=0.01, value=PASSAGE_MIN_SIMILARITY),
        html.Label(' Files to display '),
        dcc.Dropdown(id='passage-display-limit', options=[{'label': str(v), 'value': str(v)} for v in (10,25,100)] +
                     [{'label': 'All matches', 'value': 'all'}], value='25', clearable=False,
                     style={'width': '180px', 'display': 'inline-block', 'verticalAlign': 'middle'}),
        html.Button('Find text contributors', id='passage-search-button', n_clicks=0, style={'padding': '10px 16px', 'fontWeight': 'bold', 'margin': '10px'}),
    ]),
    dcc.Checklist(id='contributor-overlay-options', options=[
        {'label': ' Show contributor arrows in the tree', 'value': 'show'},
        {'label': ' Include contributor ancestor branches', 'value': 'history'},
        {'label': ' Include alternative matching files', 'value': 'alternatives'},
        {'label': ' Show intermediate assemblies', 'value': 'assemblies'},
    ], value=['show', 'history']),
    html.Label('Component groups in tree '),
    dcc.Dropdown(id='contributor-group-limit', options=[{'label': 'First 6 groups', 'value': '6'},
        {'label': 'All component groups', 'value': 'all'}], value='6', clearable=False, style={'width': '240px'}),
    html.Label('Intermediate assemblies — choose groups; empty means all when enabled'),
    dcc.Dropdown(id='intermediate-assemblies', options=[], value=[], multi=True, searchable=True),
    html.Label('Matching files explicitly included in the graph'),
    dcc.Dropdown(id='manual-matching-files', options=[], value=[], multi=True, searchable=True),
    html.P('Use Show this file in the graph beside any matching result, or select it above. Explicit files can include assemblies and otherwise redundant matches. Enable Show contributor arrows to display them.'),
    dcc.Loading(html.Div(id='passage-search-result', children='Choose a target and click Find text contributors.'), type='default'),
], open=True, style={'marginBottom': '16px', 'padding': '12px', 'border': '1px solid #b7d9c2'})]

@app.callback(Output('passage-search-result', 'children'), Output('contributor-session', 'data'),
              Input('passage-search-button', 'n_clicks'), Input('document', 'value'),
              State('passage-min-words', 'value'), State('passage-similarity', 'value'),
              State('passage-display-limit', 'value'), prevent_initial_call=True)
def update_target_passage_search(n_clicks, target_id, min_words, similarity, display_limit):
    # Switching target clears old evidence instead of leaving a misleading report.
    if dash.ctx.triggered_id != 'passage-search-button' or not n_clicks:
        return 'Target changed. Click Find text contributors to search this file.', None
    try:
        result, docs = run_target_passage_search(target_id, min_words, similarity)
        state = save_contributor_session(result, docs)
        report = render_target_passage_search(target_id, min_words, similarity, display_limit, search_data=(result, docs))
        return report, state
    except (OSError, ValueError, KeyError, TypeError, EOFError) as exc:
        return html.Div(f'Passage search unavailable: {exc}', style={'color': '#a33'}), None




@app.callback(Output('contributor-target-label', 'children'), Input('document', 'value'))
def show_contributor_target(target_id):
    if not target_id:
        return 'Target: select a document in the file selector above.'
    selected = nodes[nodes['ID'].astype(str) == str(target_id)]
    if selected.empty:
        return 'Target is unavailable in this loaded dataset.'
    row = selected.iloc[0]
    return html.Div([html.B('Target: ' + clean_text(row['Filename'])), html.Br(),
                     html.Small(clean_text(row['Full_Path'])),
                     html.P('Selecting a file shows its history. Click Find text contributors to add the matching source branches.')])

@app.callback(Output('intermediate-assemblies', 'options'), Output('intermediate-assemblies', 'value'),
              Input('contributor-session', 'data'), State('document', 'value'))
def update_intermediate_choices(state, target_id):
    try:
        entry = current_contributor_session(state, target_id)
        if entry is None:
            return [], []
        return [{'label': f"{g['representative']['filename']} | {g['representative']['target_coverage']:.1%} of target | {len(g['members'])} files",
                 'value': g['group_id']} for g in entry['result']['contributor_groups']['assembly_groups']], []
    except (OSError, ValueError):
        return [], []


@app.callback(Output('manual-matching-files', 'options'), Output('manual-matching-files', 'value'),
              Input('contributor-session', 'data'),
              Input({'type':'include-matching-file', 'source_id':ALL, 'context':ALL}, 'n_clicks'),
              State('manual-matching-files', 'value'), State('document', 'value'),
              State({'type':'include-matching-file', 'source_id':ALL, 'context':ALL}, 'id'))
def update_manual_matching_files(state, clicks, selected_ids, target_id, button_ids):
    try:
        entry = current_contributor_session(state, target_id)
        if entry is None:
            return [], []
        choices = [{'label': f"{s['filename']} | {s['target_coverage']:.1%} | {s['path']}", 'value':s['source_id']}
                   for s in entry['result']['sources']]
        allowed = {s['value'] for s in choices}
        trigger = dash.ctx.triggered_id
        if trigger == 'contributor-session':
            return choices, []
        values = [v for v in (selected_ids or []) if v in allowed]
        if isinstance(trigger, dict) and trigger.get('source_id') in allowed:
            # Insertion of new pattern buttons may trigger with zero clicks.
            actual_click = next((value for ident, value in zip(button_ids or [], clicks or []) if ident == trigger), 0)
            if isinstance(actual_click, (int, float)) and actual_click > 0:
                values = list(dict.fromkeys(values + [trigger['source_id']]))
        return choices, values
    except (OSError, ValueError):
        return [], []


@app.callback(Output({'type':'matching-file-status', 'source_id':ALL, 'context':ALL}, 'children'),
              Input('lineage-network', 'elements'), State('contributor-session', 'data'),
              State('document', 'value'), State({'type':'matching-file-status', 'source_id':ALL, 'context':ALL}, 'id'))
def show_matching_file_graph_status(elements, state, target_id, status_ids):
    try:
        entry = current_contributor_session(state, target_id)
    except (OSError, ValueError):
        entry = None
    visible = {e['data']['evidence_parent_id']:e['data'] for e in (elements or [])
               if e.get('data', {}).get('evidence_type') == 'text_contribution'}
    labels = []
    for item in status_ids or []:
        source_id = item['source_id']
        edge = visible.get(source_id)
        if edge:
            labels.append('Displayed: ' + edge.get('overlay_role', 'component') +
                          (' (explicitly included)' if edge.get('manually_included') else '') +
                          ('; alternative file' if edge.get('alternative') else ''))
        elif entry:
            grouping = entry['result']['contributor_groups']
            role = next((name for name, key in [('Intermediate assembly', 'assembly_groups'), ('Component alternative', 'selected_groups'), ('Review match', 'review_groups')]
                         if any(source_id in g['member_ids'] for g in grouping[key])), 'Matching file')
            labels.append(role + ': no physical-file arrow in this view; may be hidden by category, limits or duplicate grouping. Use Show this file in the graph to include this exact file.')
        else:
            labels.append('Search again for current graph status.')
    return labels


# Place contribution styles after normal history styles for clear parallel edges.
_network_component = next(c for c in app.layout.children if getattr(c, 'id', None) == 'lineage-network')
if hasattr(_network_component, 'stylesheet'):
    _network_component.stylesheet += [
        {'selector': '.contribution-edge', 'style': {'line-style': 'dashed', 'line-color': '#238b45',
            'target-arrow-color': '#238b45', 'width': 3, 'curve-style': 'bezier'}},
        {'selector': '.contribution-alternative', 'style': {'line-color': '#83b6a3', 'target-arrow-color': '#83b6a3', 'width': 2}},
        {'selector': '.intermediate-edge', 'style': {'line-color': '#756bb1', 'target-arrow-color': '#756bb1', 'line-style': 'dotted', 'width': 3}},
        {'selector': '.manual-match-edge', 'style': {'line-color': '#2171b5', 'target-arrow-color': '#2171b5', 'width': 3}},
        {'selector': '.text-contributor', 'style': {'border-width': 3, 'border-color': '#238b45'}},
    ]

app.run(debug=False, port=PORT)